# TASK 06 &nbsp;|&nbsp; MC Dropout uncertainty on the CNN baseline

**Due 04-Sep.** Adds Monte-Carlo dropout at inference to the Task 4 ResNet-18, reports uncertainty
estimates and calibration metrics (**ECE**, **Brier**), and exposes the result through a versioned
interface for Dr. Sattar's orchestrator.

### This notebook trains nothing

It loads Task 4's five fold checkpoints and samples them. So it needs no long GPU session — but it
does need a GPU, because MC Dropout costs `MC_PASSES` (default 50) forward passes per nodule.

### What to attach

| Dataset | Why |
|---|---|
| the **nodule cache** (`lidc_nodules_cache.npz`) | same data, same folds as Task 4 |
| the **Task 4 run's output** | contains `resnet18_fold0_best.pt` … `_fold4_best.pt` |

Both must come from the same Task 4 run. Section 6.1 checks the fingerprint stored inside every
checkpoint against this session's data and **stops** if they disagree — because if the fold
partition differs, the nodules this notebook treats as held-out include ones those models trained
on, and trained-on data always looks confident. That failure would produce good-looking numbers and
no error message, which is the worst kind.

### Where this sits

| Notebook | Produces |
|---|---|
| `task4_resnet18_cv.ipynb` | ResNet-18, 5-fold CV, checkpoints + OOF predictions |
| `task5_vit_tiny_cv.ipynb` | ViT-tiny on the identical protocol |
| **`task6_mcdropout_uq.ipynb`** | **MC Dropout UQ on the CNN baseline, + the orchestrator contract** |

Sections 4.0–4.4 below are the same shared pipeline as Tasks 4 and 5 — config, data layer, folds,
loaders, models, metrics. They are here so this notebook is self-contained and so the folds are
reconstructed identically.

### Kaggle settings

| | |
|---|---|
| Accelerator | `GPU T4 x2` |
| Internet | On (only if the cache is not attached) |
| Runtime | minutes, not hours — no training |

> A P100 will fail: recent CUDA 12.x PyTorch wheels ship no kernels for its `sm_60` architecture.
> The preflight check in the setup cell catches this in two seconds and prints the fix.

## 4.0 &nbsp;Configuration

Single source of truth for every hyperparameter in Tasks 4-6. Nothing below this cell hardcodes a
number that lives here.

In [ ]:
import os, sys, json, math, time, random, warnings, glob, zipfile, shutil, hashlib
from pathlib import Path
warnings.filterwarnings("ignore")

CFG = dict(
    SEED = 42,

    # ---------- WHICH MODEL THIS NOTEBOOK TRAINS ----------
    # TASK 6 runs MC Dropout on the CNN baseline, so this must match the Task 4 run
    # whose checkpoints you attached.
    MODEL         = "resnet18",
    PRETRAINED    = True,

    # ---------- data source ----------
    DATA_SOURCE   = "auto",        # auto | cache | lidc_raw | phantom
    CACHE_NAME    = "lidc_nodules_cache.npz",
    WORK          = "/kaggle/working",
    RAW_DIR       = "/kaggle/temp/lidc_raw",     # /kaggle/temp is not size-capped like /working
    XML_DIR       = "/kaggle/temp/lidc_xml",

    # ---------- LIDC nodule extraction ----------
    # Every value below is part of the PROTOCOL and is hashed into the dataset fingerprint.
    # Task 4 and Task 5 must agree on all of them or their results are not comparable.
    N_PATIENTS    = 300,           # 300 patients ~ 700-900 usable nodules
    MIN_READERS   = 3,             # nodule must be marked by >= this many of the 4 radiologists
    DROP_SCORE_3  = True,          # median malignancy == 3 -> ambiguous -> excluded (standard)
    MATCH_MM      = 6.0,           # cross-reader nodule matching radius
    PATCH_MM      = 48.0,          # crop is 48mm x 48mm of real anatomy, resampled to IMG_SIZE
    IMG_SIZE      = 224,
    N_SLICES      = 3,             # 1 = single axial slice, 3 = 2.5D (3 adjacent slices -> RGB)
    HU_WINDOW     = (-1000.0, 400.0),

    # ---------- patient-level cross-validation ----------
    N_FOLDS            = 5,        # Task 4 spec: patient-level 5-fold CV
    VAL_FRAC_OF_TRAIN  = 0.15,     # carved out of each fold's training pool, for early stopping

    # ---------- training ----------
    EPOCHS        = 30,
    BATCH_SIZE    = 32,
    LR            = 3e-4,
    WEIGHT_DECAY  = 1e-4,
    DROPOUT_P     = 0.30,          # TASK 6 needs this > 0 at train time
    DROPOUT_MODE  = "head_and_stages",   # head | head_and_stages
    PATIENCE      = 7,
    AMP           = True,
    NUM_WORKERS   = 2,
    CLASS_WEIGHTS = True,
    # If the attached GPU cannot run this torch build (see the preflight in the setup cell),
    # continue on CPU rather than stopping. CPU is viable at this dataset size: a measured
    # 51 min for 5 folds over 322 nodules. Results are identical, only slower.
    ALLOW_CPU_FALLBACK = True,
    # TASK 6 only: if Task 4's fold checkpoints cannot be attached, train them in Task 6 instead
    # of stopping. Same protocol (folds come from the cache + SEED), freshly trained weights.
    RETRAIN_IF_MISSING = False,

    # ---------- evaluation ----------
    TARGET_SPECIFICITIES = [0.90, 0.95, 0.99],
    N_BOOTSTRAP   = 1000,
    N_ECE_BINS    = 15,

    # ---------- TASK 6 : MC dropout ----------
    MC_PASSES     = 50,
    UQ_TARGET_SENSITIVITY = 0.98,
)

CFG["OUT_DIR"]    = f"{CFG['WORK']}/{CFG['MODEL']}_task_outputs"
CFG["CACHE_PATH"] = f"{CFG['WORK']}/{CFG['CACHE_NAME']}"
Path(CFG["OUT_DIR"]).mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["benign", "malignant"]
IS_SMOKE = False          # flipped True by the phantom fallback; gates all reporting

# Keys that define the protocol. Both notebooks must match on these, so they are hashed into
# the fingerprint that ships with the results bundle.
PROTOCOL_KEYS = ["SEED", "N_PATIENTS", "MIN_READERS", "DROP_SCORE_3", "MATCH_MM", "PATCH_MM",
                 "IMG_SIZE", "N_SLICES", "HU_WINDOW", "N_FOLDS", "VAL_FRAC_OF_TRAIN"]

print(f"This notebook does NOT train. It runs MC Dropout on attached "
      f"{CFG['MODEL']} fold checkpoints with T={CFG['MC_PASSES']} passes.")
print(json.dumps(CFG, indent=2, default=str))

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False   # keep benchmark speed; seeds still control init/sampling
    torch.backends.cudnn.benchmark = True

set_seed(CFG["SEED"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

# timm supplies the ViT for Task 5. Preinstalled on recent Kaggle images.
try:
    import timm
    print("timm", timm.__version__)
except ImportError:
    get_ipython().system("pip install -q timm")
    import timm
    print("timm", timm.__version__, "(freshly installed)")


def gpu_preflight():
    """Prove the GPU can actually run a convolution BEFORE anything expensive happens.

    A PyTorch wheel only ships kernels for the GPU architectures it was compiled for. If the
    attached accelerator is not in that list, every CUDA op raises "no kernel image is available
    for execution on the device" - but only when first called, which is inside the training loop,
    which here is after a multi-hour download. One tiny convolution at setup time turns a wasted
    session into a two-second message.

    On failure: falls back to CPU if CFG["ALLOW_CPU_FALLBACK"], otherwise stops the run.
    CPU is genuinely viable at this dataset size - a measured 51 min for 5 folds on 322 nodules -
    so the fallback is a real option, not a token one.
    """
    global DEVICE

    def cpu_estimate():
        n = len(LABELS) if "LABELS" in globals() else CFG.get("N_PATIENTS", 300)
        # ~51 min measured for 5 folds over 322 nodules at 224px on Kaggle CPU
        return max(5, round(51 * (n / 322.0)))

    if not torch.cuda.is_available():
        print("")
        print("No accelerator attached - running on CPU.")
        print(f"  At this dataset size expect roughly {cpu_estimate()} min for all "
              f"{CFG['N_FOLDS']} folds.")
        print("  For a GPU: Kaggle sidebar -> Session options -> Accelerator -> GPU T4 x2")
        print("  (Do NOT pick P100 - see below.)")
        print("")
        return False

    name = torch.cuda.get_device_name(0)
    major, minor = torch.cuda.get_device_capability(0)
    mine = f"sm_{major}{minor}"
    built_for = [a for a in torch.cuda.get_arch_list() if a.startswith("sm_")]
    print("")
    print(f"GPU               : {name}")
    print(f"compute capability: {mine}")
    print(f"torch built for   : {', '.join(built_for) or 'unknown'}")

    try:
        x = torch.zeros(1, 3, 8, 8, device="cuda")
        w = torch.zeros(4, 3, 3, 3, device="cuda")
        torch.nn.functional.conv2d(x, w)
        torch.cuda.synchronize()
        print("preflight         : OK - a real convolution ran on the device")
        print("")
        return True
    except Exception as e:
        report = f"""
==========================================================================
THIS GPU CANNOT RUN THIS PYTORCH BUILD
==========================================================================
  GPU              : {name}  ({mine})
  torch {torch.__version__}
  has kernels for  : {', '.join(built_for)}
  underlying error : {type(e).__name__}: {e}

  {mine} is not in that list, so every CUDA operation on this device fails.
  This is an environment mismatch, not a bug in this notebook.

  BEST FIX: Kaggle sidebar -> Session options -> Accelerator -> "GPU T4 x2"
            The T4 is sm_75 and is covered by this build. Then Run All again.
            (A P100 is sm_60 - Pascal - which recent CUDA 12.x wheels no
             longer ship kernels for.)
=========================================================================="""
        print(report)

        if CFG.get("ALLOW_CPU_FALLBACK", True):
            DEVICE = torch.device("cpu")
            print(f"""
  ALLOW_CPU_FALLBACK is on, so this run CONTINUES ON CPU.
  Expect roughly {cpu_estimate()} min for all {CFG['N_FOLDS']} folds - slower than a
  working GPU, but it finishes, and results are identical.
  Set CFG["ALLOW_CPU_FALLBACK"] = False to stop here instead.
==========================================================================
""")
            return False

        raise RuntimeError(report + """

  Or set CFG["ALLOW_CPU_FALLBACK"] = True to continue on CPU instead.
""") from e


GPU_OK = gpu_preflight()
print("training device   :", DEVICE)

## 4.1 &nbsp;Data layer - building labelled nodules from LIDC-IDRI

### 4.1.1 &nbsp;Why this section exists

The LIDC-IDRI label is not a column in a CSV. It is produced by up to four radiologists who each
independently outlined every nodule they saw and scored it on nine characteristics, malignancy
being one of them (1 = highly unlikely, 5 = highly suspicious). Turning that into a training label
requires three decisions, and every LIDC paper makes them - the numbers are only comparable
between papers that made them the same way. Ours, matching the dominant convention:

| Decision | Choice | Why |
|---|---|---|
| Which nodules to keep | marked by `MIN_READERS` = 3+ radiologists | 1-reader nodules are noisy and inflate scores |
| How to fuse the 4 ratings | **median** | robust to one dissenting reader; standard in the literature |
| What to do with median == 3 | **drop** | "indeterminate" is neither benign nor malignant; keeping it corrupts both classes |

These are recorded in `CFG` so the exact protocol is reproducible and quotable in the paper.
The other eight characteristics (spiculation, margin, subtlety, lobulation, calcification,
texture, sphericity, internal structure) are cached too - Task 8's concept-based explainability
(TCAV/ACE) needs them, and re-extracting later would mean re-downloading everything.

In [ ]:
import xml.etree.ElementTree as ET
from collections import defaultdict

def _tag(elem):
    """LIDC XMLs carry a namespace that varies between files. Compare on the local name only."""
    return elem.tag.split("}")[-1]

CHARACTERISTIC_KEYS = ["subtlety", "internalStructure", "calcification", "sphericity",
                       "margin", "lobulation", "spiculation", "texture", "malignancy"]

def parse_lidc_xml(xml_path):
    """One LIDC XML -> (series_uid, [reading_session, ...]).

    Each reading_session is one radiologist's list of nodules; each nodule carries its
    characteristic scores and its per-slice contours. Nodules with no <characteristics>
    block are the <3mm marks, which have no malignancy rating - dropped here.
    """
    root = ET.parse(xml_path).getroot()

    series_uid = None
    for e in root.iter():
        if _tag(e) in ("SeriesInstanceUid", "SeriesInstanceUID"):
            series_uid = (e.text or "").strip()
            break

    sessions = []
    for sess in root.iter():
        if _tag(sess) != "readingSession":
            continue
        nodules = []
        for nod in sess:
            if _tag(nod) != "unblindedReadNodule":
                continue
            nid, chars, rois = None, {}, []
            for child in nod:
                t = _tag(child)
                if t == "noduleID":
                    nid = (child.text or "").strip()
                elif t == "characteristics":
                    for c in child:
                        try:
                            chars[_tag(c)] = int(float((c.text or "").strip()))
                        except (TypeError, ValueError):
                            pass
                elif t == "roi":
                    z, sop, pts, included = None, None, [], True
                    for r in child:
                        rt = _tag(r)
                        if rt == "imageZposition":
                            z = float(r.text)
                        elif rt == "imageSOP_UID":
                            sop = (r.text or "").strip()
                        elif rt == "inclusion":
                            included = (r.text or "").strip().upper() == "TRUE"
                        elif rt == "edgeMap":
                            x = y = None
                            for e2 in r:
                                if _tag(e2) == "xCoord":   x = int(float(e2.text))
                                elif _tag(e2) == "yCoord": y = int(float(e2.text))
                            if x is not None and y is not None:
                                pts.append((x, y))
                    if pts and included:
                        rois.append(dict(z=z, sop=sop, pts=np.asarray(pts, dtype=np.float32)))
            # no malignancy score -> unusable as a label, however good the contour
            if rois and "malignancy" in chars:
                nodules.append(dict(nodule_id=nid, characteristics=chars, rois=rois))
        sessions.append(nodules)

    return series_uid, sessions


def nodule_geometry(nodule):
    """Centroid as (col, row, z_mm) plus the largest in-plane diameter, in pixels."""
    all_pts = np.concatenate([r["pts"] for r in nodule["rois"]], axis=0)
    cx, cy = all_pts[:, 0].mean(), all_pts[:, 1].mean()
    zs = [r["z"] for r in nodule["rois"] if r["z"] is not None]
    cz = float(np.mean(zs)) if zs else None
    diam_px = 0.0
    for r in nodule["rois"]:
        p = r["pts"]
        if len(p) > 1:
            diam_px = max(diam_px, float(np.hypot(np.ptp(p[:, 0]), np.ptp(p[:, 1]))))
    return float(cx), float(cy), cz, diam_px

### 4.1.2 &nbsp;Matching the same nodule across radiologists

The four readers worked blind to each other, so reader 1's "nodule 3" and reader 4's "nodule 1" may
be the same lesion. They must be grouped before the ratings can be fused. Grouping is by spatial
proximity of centroids in millimetres, with one constraint a naive nearest-neighbour pass would get
wrong: **a cluster may contain at most one nodule per reader.** Without it, two genuinely distinct
lesions that a single reader marked close together collapse into one cluster and its reader count
becomes meaningless - which then silently breaks the `MIN_READERS` filter.

In [ ]:
def cluster_nodules_across_readers(sessions, spacing_xy, match_mm):
    """Group per-reader nodule marks into physical lesions.

    Greedy agglomeration on centroid distance in mm, with the one-nodule-per-reader constraint.
    Returns a list of clusters; each cluster is a list of per-reader mark records.
    """
    items = []
    for r_idx, nodules in enumerate(sessions):
        for nod in nodules:
            cx, cy, cz, diam = nodule_geometry(nod)
            if cz is None:
                continue
            items.append(dict(reader=r_idx, nodule=nod, cx=cx, cy=cy, cz=cz, diam_px=diam))

    # Deterministic order so the same input always yields the same clusters.
    items.sort(key=lambda d: (d["cz"], d["cx"], d["cy"], d["reader"]))
    sx, sy = spacing_xy
    clusters = []

    for it in items:
        best, best_d = None, np.inf
        for cl in clusters:
            if any(m["reader"] == it["reader"] for m in cl):
                continue                                   # this reader is already represented
            mcx = np.mean([m["cx"] for m in cl]); mcy = np.mean([m["cy"] for m in cl])
            mcz = np.mean([m["cz"] for m in cl])
            d = math.sqrt(((it["cx"] - mcx) * sx) ** 2 +
                          ((it["cy"] - mcy) * sy) ** 2 +
                          (it["cz"] - mcz) ** 2)
            if d < best_d:
                best, best_d = cl, d
        if best is not None and best_d <= match_mm:
            best.append(it)
        else:
            clusters.append([it])
    return clusters


def fuse_cluster_labels(cluster, min_readers, drop_score_3):
    """Cluster of per-reader marks -> one labelled nodule, or None if it fails the protocol."""
    readers = sorted({m["reader"] for m in cluster})
    if len(readers) < min_readers:
        return None

    scores = defaultdict(list)
    for m in cluster:
        for k in CHARACTERISTIC_KEYS:
            if k in m["nodule"]["characteristics"]:
                scores[k].append(m["nodule"]["characteristics"][k])
    if not scores.get("malignancy"):
        return None

    mal_median = float(np.median(scores["malignancy"]))
    if drop_score_3 and abs(mal_median - 3.0) < 1e-9:
        return None
    label = int(mal_median > 3.0)

    return dict(
        n_readers   = len(readers),
        malignancy  = mal_median,
        label       = label,
        # Disagreement between readers. Task 6 uses this as a reference for irreducible
        # (aleatoric) label noise: cases the radiologists split on are cases the model
        # should be uncertain about too.
        mal_std     = float(np.std(scores["malignancy"])),
        mal_ratings = ",".join(str(int(v)) for v in scores["malignancy"]),
        cx = float(np.mean([m["cx"] for m in cluster])),
        cy = float(np.mean([m["cy"] for m in cluster])),
        cz = float(np.mean([m["cz"] for m in cluster])),
        diam_px = float(np.mean([m["diam_px"] for m in cluster])),
        **{f"c_{k}": float(np.median(v)) for k, v in scores.items()},
    )

### 4.1.3 &nbsp;Reading the CT series and cutting patches

Two details that matter more than they look:

**Patches are sized in millimetres, not pixels.** Pixel spacing across the 3 patients Task 3
already pulled ranges 0.664 - 0.781 mm. A fixed 64x64-pixel crop would therefore contain ~18% more
anatomy on one scan than another, and the network would be partly learning scanner settings. So we
crop `PATCH_MM` = 48 mm of real tissue and resample to `IMG_SIZE`, making nodule size physically
comparable across patients.

**`N_SLICES` = 3 gives a 2.5D input for free.** Three adjacent axial slices stacked as three
channels feed an ImageNet-pretrained 3-channel stem with no surgery, and carry some through-plane
context a single slice does not. Set it to 1 for a strict 2D baseline.

In [ ]:
def load_ct_series(series_dir):
    """Read one DICOM CT series -> HU volume sorted along z, plus geometry."""
    import pydicom
    paths = glob.glob(os.path.join(series_dir, "**", "*.dcm"), recursive=True)
    slices = []
    for p in paths:
        try:
            ds = pydicom.dcmread(p, force=True)
            if not hasattr(ds, "PixelData") or not hasattr(ds, "ImagePositionPatient"):
                continue
            if getattr(ds, "Modality", "CT") != "CT":
                continue
            slices.append(ds)
        except Exception:
            continue
    if len(slices) < 10:
        return None

    slices.sort(key=lambda d: float(d.ImagePositionPatient[2]))
    ref = slices[0]
    slope     = float(getattr(ref, "RescaleSlope", 1.0))
    intercept = float(getattr(ref, "RescaleIntercept", 0.0))
    spacing   = [float(v) for v in getattr(ref, "PixelSpacing", [1.0, 1.0])]

    vol = np.stack([s.pixel_array.astype(np.float32) for s in slices], axis=0)
    vol = vol * slope + intercept                   # stored values -> Hounsfield Units
    vol[vol < -2000] = -1000.0                      # outside-the-bore padding -> air

    return dict(
        vol       = vol,                                              # (z, y, x) HU
        z_pos     = np.array([float(s.ImagePositionPatient[2]) for s in slices]),
        spacing   = (spacing[1], spacing[0]),                         # (x, y) mm per pixel
        thickness = float(getattr(ref, "SliceThickness", 1.0)),
        n_slices  = len(slices),
        patient_id= str(getattr(ref, "PatientID", "")),
        series_uid= str(getattr(ref, "SeriesInstanceUID", "")),
    )


def window_to_uint8(patch_hu, hu_window):
    lo, hi = hu_window
    x = (np.clip(patch_hu, lo, hi) - lo) / (hi - lo)
    return (x * 255.0).astype(np.uint8)


def extract_patch(series, nod, patch_mm, img_size, n_slices, hu_window):
    """Nodule centroid -> (n_slices, img_size, img_size) uint8 patch, or None."""
    import cv2
    vol, z_pos = series["vol"], series["z_pos"]
    sx, sy = series["spacing"]

    z_idx  = int(np.argmin(np.abs(z_pos - nod["cz"])))
    half_x = max(2, int(round((patch_mm / 2.0) / sx)))
    half_y = max(2, int(round((patch_mm / 2.0) / sy)))
    cx, cy = int(round(nod["cx"])), int(round(nod["cy"]))

    offsets = [0] if n_slices == 1 else list(range(-(n_slices // 2), n_slices // 2 + 1))
    out = []
    for off in offsets:
        zi = int(np.clip(z_idx + off, 0, vol.shape[0] - 1))
        # Pad first, so a nodule against the chest wall still yields a full-size CENTRED crop.
        # Cropping first and padding after would shift the nodule off-centre.
        pad_y, pad_x = half_y + 1, half_x + 1
        padded = np.pad(vol[zi], ((pad_y, pad_y), (pad_x, pad_x)),
                        mode="constant", constant_values=hu_window[0])
        py, px = cy + pad_y, cx + pad_x
        crop = padded[py - half_y: py + half_y, px - half_x: px + half_x]
        if crop.size == 0:
            return None
        out.append(cv2.resize(window_to_uint8(crop, hu_window), (img_size, img_size),
                              interpolation=cv2.INTER_LINEAR))
    return np.stack(out, axis=0)

### 4.1.4 &nbsp;Fetching raw LIDC-IDRI (DICOM **and** the XML annotations Task 3 was missing)

`tcia_utils` handles the images. The annotations are a separate ~50 MB package. If the URL has moved
(TCIA reorganises its downloads periodically), the cell fails with printed instructions instead of
silently continuing with no labels - which is exactly the failure mode that produced the empty
`no annotations found` result in Task 3.

In [ ]:
def download_lidc_dicom(n_patients, out_dir):
    """Download one CT series per patient for the first n_patients (deterministic selection)."""
    try:
        from tcia_utils import nbia
    except ImportError:
        get_ipython().system("pip install -q tcia_utils")
        from tcia_utils import nbia

    os.makedirs(out_dir, exist_ok=True)
    df = pd.DataFrame(nbia.getSeries(collection="LIDC-IDRI"))
    df = df[df["Modality"] == "CT"].copy()               # LIDC also contains DX/CR radiographs
    df["ImageCount"] = pd.to_numeric(df["ImageCount"], errors="coerce").fillna(0)

    # One series per patient: the largest stack, i.e. the diagnostic CT rather than a scout.
    df = (df.sort_values(["PatientID", "ImageCount"], ascending=[True, False])
            .groupby("PatientID", as_index=False).first()
            .sort_values("PatientID"))
    sel = df.head(n_patients)
    print(f"LIDC-IDRI: {df.PatientID.nunique()} patients with CT | selecting {len(sel)}")
    print(f"Estimated download: ~{0.9 * sel.ImageCount.sum() / 1000:.1f} GB "
          f"({int(sel.ImageCount.sum())} slices)")

    nbia.downloadSeries(sel.to_dict("records"), path=out_dir)
    return sel


LIDC_XML_URLS = [
    "https://www.cancerimagingarchive.net/wp-content/uploads/LIDC-XML-only.zip",
    "https://wiki.cancerimagingarchive.net/download/attachments/1966254/LIDC-XML-only.zip",
]

def download_lidc_xml(xml_dir, urls=LIDC_XML_URLS):
    """Fetch + unzip the LIDC XML annotation package. Raises loudly if unavailable."""
    import requests
    os.makedirs(xml_dir, exist_ok=True)
    if glob.glob(os.path.join(xml_dir, "**", "*.xml"), recursive=True):
        print("XML annotations already present, skipping download.")
        return xml_dir

    zip_path = os.path.join(xml_dir, "LIDC-XML-only.zip")
    for url in urls:
        try:
            print("Trying", url)
            with requests.get(url, stream=True, timeout=180) as r:
                r.raise_for_status()
                with open(zip_path, "wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(xml_dir)
            os.remove(zip_path)
            n = len(glob.glob(os.path.join(xml_dir, "**", "*.xml"), recursive=True))
            print(f"Extracted {n} annotation XMLs.")
            return xml_dir
        except Exception as e:
            print("  failed:", type(e).__name__, e)

    raise RuntimeError(
        "Could not fetch the LIDC XML annotation package automatically.\n"
        "MANUAL FIX (5 minutes, needed once):\n"
        "  1. Open the LIDC-IDRI collection page on cancerimagingarchive.net\n"
        "  2. Under Data Access / supporting documents, download 'LIDC-XML-only.zip'\n"
        "  3. Upload it as a Kaggle Dataset and attach it to this notebook\n"
        "  4. Set CFG['XML_DIR'] to that /kaggle/input/... path and re-run this cell\n"
        "Without these XMLs there are no malignancy labels and no supervised task."
    )


def index_xml_by_series(xml_dir):
    """SeriesInstanceUID -> xml path, so each downloaded series can find its annotations."""
    idx, files = {}, glob.glob(os.path.join(xml_dir, "**", "*.xml"), recursive=True)
    for p in files:
        try:
            uid, _ = parse_lidc_xml(p)
            if uid:
                idx[uid] = p
        except Exception:
            continue
    print(f"Indexed {len(idx)} / {len(files)} XMLs by SeriesInstanceUID.")
    return idx

In [ ]:
def build_lidc_cache(cfg):
    """Raw LIDC -> labelled-patch cache. Run ONCE, then save the .npz as a Kaggle Dataset."""
    import pydicom

    t0 = time.time()
    download_lidc_dicom(cfg["N_PATIENTS"], cfg["RAW_DIR"])
    xml_index = index_xml_by_series(download_lidc_xml(cfg["XML_DIR"]))

    patches, rows = [], []
    stats = defaultdict(int)
    # sorted(): traversal order must not influence the cache
    series_dirs = sorted(d for d in glob.glob(os.path.join(cfg["RAW_DIR"], "**"), recursive=True)
                         if os.path.isdir(d) and glob.glob(os.path.join(d, "*.dcm")))
    print(f"Found {len(series_dirs)} series directories containing DICOM.")

    for i, sdir in enumerate(series_dirs, 1):
        series = load_ct_series(sdir)
        if series is None:
            stats["series_unreadable"] += 1
            continue

        suid = series["series_uid"] or os.path.basename(sdir)
        if suid not in xml_index:
            stats["series_no_annotation"] += 1
            continue

        _, sessions = parse_lidc_xml(xml_index[suid])
        clusters = cluster_nodules_across_readers(sessions, series["spacing"], cfg["MATCH_MM"])
        stats["clusters_found"] += len(clusters)

        kept = 0
        for cl in clusters:
            nod = fuse_cluster_labels(cl, cfg["MIN_READERS"], cfg["DROP_SCORE_3"])
            if nod is None:
                stats["rejected_by_protocol"] += 1
                continue
            patch = extract_patch(series, nod, cfg["PATCH_MM"], cfg["IMG_SIZE"],
                                  cfg["N_SLICES"], cfg["HU_WINDOW"])
            if patch is None:
                stats["patch_failed"] += 1
                continue
            nod["patient_id"] = series["patient_id"] or suid
            nod["series_uid"] = suid
            nod["diam_mm"]    = nod["diam_px"] * float(np.mean(series["spacing"]))
            patches.append(patch); rows.append(nod); kept += 1

        stats["series_used"] += 1
        if i % 25 == 0 or kept:
            print(f"  [{i}/{len(series_dirs)}] {series['patient_id']}: +{kept} nodules "
                  f"(total {len(patches)})")

        del series          # a 512x512x140 float32 volume is ~150 MB; do not accumulate them

    if not patches:
        raise RuntimeError(f"No nodules extracted. Diagnostics: {dict(stats)}")

    # ---- canonical row order ----
    # The fold assignment is derived from row order, so row order must be a function of the DATA,
    # not of how the files happened to be laid out on disk. Sort by patient, then series, then
    # nodule position. Two independent downloads of the same patients now yield byte-identical
    # caches and therefore the same fingerprint.
    order = sorted(range(len(rows)), key=lambda i: (
        str(rows[i]["patient_id"]), str(rows[i]["series_uid"]),
        round(float(rows[i]["cz"]), 2), round(float(rows[i]["cy"]), 2),
        round(float(rows[i]["cx"]), 2)))
    patches = [patches[i] for i in order]
    rows    = [rows[i] for i in order]

    X    = np.stack(patches).astype(np.uint8)
    meta = pd.DataFrame(rows)
    np.savez_compressed(
        cfg["CACHE_PATH"],
        patches    = X,
        labels     = meta["label"].to_numpy(np.int64),
        malignancy = meta["malignancy"].to_numpy(np.float32),
        patient_id = meta["patient_id"].to_numpy(object),
        meta_json  = json.dumps(meta.to_dict("list"), default=str),
        cfg_json   = json.dumps({k: cfg[k] for k in
                                 ["MIN_READERS", "DROP_SCORE_3", "MATCH_MM", "PATCH_MM",
                                  "IMG_SIZE", "N_SLICES", "HU_WINDOW", "N_PATIENTS"]},
                                default=str),
    )
    print(f"\nBuild finished in {(time.time() - t0) / 60:.1f} min")
    print(f"  {X.shape[0]} nodules from {meta.patient_id.nunique()} patients")
    print(f"  cache -> {cfg['CACHE_PATH']}  ({os.path.getsize(cfg['CACHE_PATH'])/1e6:.0f} MB)")
    print(f"  diagnostics: {dict(stats)}")
    print("\n>>> Save this notebook version, then add its output .npz as a Kaggle Dataset")
    print(">>> so Tasks 4/5/6 never have to re-download.")
    return cfg["CACHE_PATH"]

### 4.1.5 &nbsp;Phantom fallback - `SMOKE` only

If no cache exists and the download cannot run, this generates synthetic lung-window patches so
Sections 2-6 can be executed and debugged. It sets a global `IS_SMOKE` flag and every results table
downstream is stamped with it.

**A `SMOKE` number proves the code runs. It is not a result.** Do not put one in the paper, the
Google Sheet, or an email to Dr. Sattar.

In [ ]:
def build_phantom_cache(cfg, n_patients=60, per_patient=12):
    """Synthetic stand-in: benign = round + smooth, malignant = irregular + spiculated."""
    import cv2
    rng = np.random.default_rng(cfg["SEED"])
    S, K = cfg["IMG_SIZE"], cfg["N_SLICES"]
    X, labels, pids, mal = [], [], [], []

    for p in range(n_patients):
        pid = f"PHANTOM-{p:04d}"
        for _ in range(per_patient):
            y = int(rng.random() < 0.40)
            base = cv2.GaussianBlur(rng.normal(0.25, 0.06, (S, S)).astype(np.float32), (0, 0), 6.0)
            yy, xx = np.mgrid[0:S, 0:S].astype(np.float32)
            cxp, cyp = S / 2 + rng.normal(0, 4, 2)
            r = S * rng.uniform(0.10, 0.20)
            theta = np.arctan2(yy - cyp, xx - cxp)
            k, amp = int(rng.integers(6, 12)), (0.32 if y else 0.06)   # spiculation proxy
            rr = r * (1.0 + amp * np.sin(k * theta + rng.uniform(0, 6.28)))
            blob = np.clip((rr - np.hypot(xx - cxp, yy - cyp)) / (0.18 * r), 0, 1)
            img = base + blob * rng.uniform(0.45, 0.70)
            if y:
                img += rng.normal(0, 0.05, (S, S)) * blob              # heterogeneous texture
            img = np.clip(cv2.GaussianBlur(img, (0, 0), 1.0), 0, 1)
            vol = np.stack([np.clip(img + rng.normal(0, 0.012, (S, S)), 0, 1) for _ in range(K)])
            X.append((vol * 255).astype(np.uint8))
            labels.append(y); pids.append(pid)
            mal.append(float(rng.integers(4, 6) if y else rng.integers(1, 3)))

    np.savez_compressed(cfg["CACHE_PATH"], patches=np.stack(X),
                        labels=np.asarray(labels, np.int64),
                        malignancy=np.asarray(mal, np.float32),
                        patient_id=np.asarray(pids, object),
                        meta_json=json.dumps({}), cfg_json=json.dumps({"PHANTOM": True}))
    print(f"PHANTOM cache: {len(X)} patches / {n_patients} pseudo-patients -> {cfg['CACHE_PATH']}")
    return cfg["CACHE_PATH"]

### 4.1.6 &nbsp;Source resolution

Preference order: an attached Kaggle Dataset cache, then a cache written earlier in this session,
then a fresh raw build, then the phantom. Override with `CFG["DATA_SOURCE"]`.

In [ ]:
def find_input_cache(cache_name):
    """Search attached Kaggle Datasets for a prebuilt cache with the right keys."""
    hits = glob.glob(f"/kaggle/input/**/{cache_name}", recursive=True)
    hits += [p for p in glob.glob("/kaggle/input/**/*.npz", recursive=True) if p not in hits]
    for p in hits:
        try:
            with np.load(p, allow_pickle=True) as z:
                if {"patches", "labels", "patient_id"} <= set(z.files):
                    return p
        except Exception:
            continue
    return None


def resolve_data(cfg):
    global IS_SMOKE
    src = cfg["DATA_SOURCE"]

    if src in ("auto", "cache"):
        p = find_input_cache(cfg["CACHE_NAME"])
        if p:
            print("Using cache from attached dataset:", p)
            return p
        if os.path.exists(cfg["CACHE_PATH"]):
            print("Using cache from this session:", cfg["CACHE_PATH"])
            return cfg["CACHE_PATH"]
        if src == "cache":
            raise FileNotFoundError("DATA_SOURCE='cache' but no cache was found.")

    if src in ("auto", "lidc_raw"):
        try:
            return build_lidc_cache(cfg)
        except Exception as e:
            print("\n" + "!" * 78)
            print("RAW LIDC BUILD FAILED:", type(e).__name__)
            print(e)
            print("!" * 78 + "\n")
            if src == "lidc_raw":
                raise

    print("\n" + "=" * 78)
    print("FALLING BACK TO SYNTHETIC PHANTOM DATA - RESULTS ARE NOT REPORTABLE")
    print("=" * 78 + "\n")
    IS_SMOKE = True
    return build_phantom_cache(cfg)


CACHE = resolve_data(CFG)

In [ ]:
with np.load(CACHE, allow_pickle=True) as z:
    PATCHES    = z["patches"]                          # (N, K, S, S) uint8
    LABELS     = z["labels"].astype(np.int64)
    MALIGNANCY = z["malignancy"].astype(np.float32)
    PATIENT_ID = np.asarray([str(v) for v in z["patient_id"]])
    BUILD_CFG  = json.loads(str(z["cfg_json"])) if "cfg_json" in z.files else {}
    _mj        = str(z["meta_json"]) if "meta_json" in z.files else "{}"
META = pd.DataFrame(json.loads(_mj)) if _mj not in ("{}", "", "None") else pd.DataFrame()

IS_SMOKE = IS_SMOKE or bool(BUILD_CFG.get("PHANTOM", False))

# Hash of the patient set alone. If two runs disagree on the FINGERPRINT, this says whether the
# cause was a different set of patients (a download that dropped a series) or something else -
# which decides whether you need to rebuild or only re-run the training.
PATIENT_SET = sorted(set(PATIENT_ID.tolist()))
PATIENT_SET_HASH = hashlib.sha256("|".join(PATIENT_SET).encode()).hexdigest()[:16]

print(f"patches      : {PATCHES.shape}  {PATCHES.dtype}  ({PATCHES.nbytes/1e6:.0f} MB in RAM)")
print(f"patients     : {len(np.unique(PATIENT_ID))}")
print(f"nodules      : {len(LABELS)}")
print(f"class balance: benign={int((LABELS==0).sum())}  malignant={int((LABELS==1).sum())} "
      f"({100*LABELS.mean():.1f}% positive)")
print(f"build config : {BUILD_CFG}")
print(f"patient set  : {len(PATIENT_SET)} patients, hash {PATIENT_SET_HASH}")
print(f"SMOKE MODE   : {IS_SMOKE}")
if len(META):
    print(f"characteristics cached for Task 8: "
          f"{[c for c in META.columns if c.startswith('c_')]}")

# A baseline trained on a handful of nodules gives a meaningless AUC with an enormous CI.
if not IS_SMOKE and len(LABELS) < 200:
    print(f"\nWARNING: only {len(LABELS)} nodules. Raise CFG['N_PATIENTS'] and rebuild before "
          f"reporting anything - patient-level confidence intervals will be uninterpretably wide.")

### 4.1.7 &nbsp;Patient-level 5-fold cross-validation

The task specifies **patient-level 5-fold CV**, and both words matter.

**5-fold, not a single split.** One held-out test set of ~150 nodules gives an AUROC whose
confidence interval is far too wide to distinguish two architectures. Cross-validation instead
produces an out-of-fold (OOF) prediction for *every* nodule in the dataset: each nodule is scored
exactly once, by the one fold model that never saw its patient during training. That gives 5x the
evaluation data at no extra data cost, plus a per-fold spread that tells you how stable the result
actually is.

**Patient-level, not nodule-level.** One patient contributes several nodules from the same scan,
scanner and reconstruction kernel. Splitting on nodules lets the same patient appear in both train
and test, which can add several points of pure illusion to the AUROC. Grouping is by
`patient_id` everywhere, and the assertions below **fail the run** rather than letting a leak pass
quietly.

Each fold gets three disjoint patient groups: the held-out fold is the test set, and a validation
slice is carved out of the remaining four folds for early stopping. Selecting the epoch on the test
fold would leak it, so validation is never the fold being scored.

**Why the fingerprint matters here.** You are running Tasks 4 and 5 on separate Kaggle accounts, so
nothing structurally forces the two runs to use the same data. The fold assignment is derived from
the label vector, the patient IDs and `SEED` - so identical inputs give identical folds, and the
fingerprint printed below is a short hash of exactly those inputs. If the two notebooks print the
**same fingerprint**, they were scored on the same nodules in the same folds and their results can
be compared with paired statistics. If they differ, they cannot. Check it; do not assume it.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

def patient_level_folds(labels, groups, n_folds, val_frac_of_train, seed):
    """Patient-level, stratified k-fold. Returns one dict per fold with disjoint train/val/test.

    StratifiedGroupKFold keeps whole patients together AND balances the class ratio across folds.
    The validation slice is taken from the training pool only, never from the fold being scored.
    """
    idx_all = np.arange(len(labels))
    outer = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)

    folds = []
    for k, (tr_pool, test_idx) in enumerate(outer.split(idx_all, labels, groups)):
        k_val = max(2, int(round(1.0 / val_frac_of_train)))
        sub_tr, sub_val = next(StratifiedGroupKFold(
            n_splits=k_val, shuffle=True, random_state=seed + 100 + k
        ).split(tr_pool, labels[tr_pool], groups[tr_pool]))
        folds.append(dict(fold=k, train=tr_pool[sub_tr], val=tr_pool[sub_val], test=test_idx))

    # ---- correctness assertions: a silent leak here invalidates every number downstream ----
    for f in folds:
        for a, b, what in [("train", "val", "train/val"), ("train", "test", "train/test"),
                           ("val", "test", "val/test")]:
            overlap = set(groups[f[a]]) & set(groups[f[b]])
            assert not overlap, (f"PATIENT LEAK in fold {f['fold']} between {what}: "
                                 f"{sorted(overlap)[:5]}")
    covered = np.concatenate([f["test"] for f in folds])
    assert len(covered) == len(np.unique(covered)) == len(labels), (
        "test folds must partition the dataset exactly once per nodule")
    return folds


def dataset_fingerprint(labels, groups, folds, cfg):
    """Short hash of everything that must match for a paired comparison to be valid:
    the labels, the patient assignment, the protocol constants, and the fold split."""
    h = hashlib.sha256()
    h.update(np.asarray(labels, dtype=np.int64).tobytes())
    h.update("|".join(map(str, groups)).encode())
    for k in PROTOCOL_KEYS:
        h.update(f"{k}={cfg[k]}".encode())
    fold_of = np.full(len(labels), -1, dtype=np.int64)
    for f in folds:
        fold_of[f["test"]] = f["fold"]
    h.update(fold_of.tobytes())
    return h.hexdigest()[:16], fold_of


FOLDS = patient_level_folds(LABELS, PATIENT_ID, CFG["N_FOLDS"],
                            CFG["VAL_FRAC_OF_TRAIN"], CFG["SEED"])
FINGERPRINT, FOLD_OF = dataset_fingerprint(LABELS, PATIENT_ID, FOLDS, CFG)

FOLD_TABLE = pd.DataFrame([
    dict(fold=f["fold"],
         train_nodules=len(f["train"]), val_nodules=len(f["val"]), test_nodules=len(f["test"]),
         train_patients=len(np.unique(PATIENT_ID[f["train"]])),
         val_patients=len(np.unique(PATIENT_ID[f["val"]])),
         test_patients=len(np.unique(PATIENT_ID[f["test"]])),
         test_pct_malignant=round(100 * float(LABELS[f["test"]].mean()), 1))
    for f in FOLDS])
print(FOLD_TABLE.to_string(index=False))
FOLD_TABLE.to_csv(f"{CFG['OUT_DIR']}/fold_table.csv", index=False)

print(f"\nNo patient appears in more than one split of any fold (asserted).")
print(f"Every nodule is in exactly one test fold (asserted).")
print("\n" + "=" * 78)
print(f"  DATASET FINGERPRINT : {FINGERPRINT}")
print("=" * 78)
print("  Task 4 and Task 5 MUST print the same fingerprint for a paired comparison to be")
print("  valid. If they differ, the two runs used different data or different folds, and")
print("  DeLong / McNemar cannot be applied to them.")
print("=" * 78)

### 4.1.8 &nbsp;Dataset, augmentation, per-fold loaders

Augmentation is deliberately restrained. Flips and 90-degree rotations are safe on an axial lung
patch - a nodule has no canonical orientation. Aggressive intensity jitter is **not** safe: the
input is calibrated Hounsfield Units, and distorting them destroys the density information that
separates a solid nodule from a ground-glass one. Jitter is therefore kept small, and that choice
is a methodological claim worth stating in the paper rather than a default left unexamined.

Loaders are built **per fold**, and only the training loader augments or rebalances. The validation
and test loaders see the data untouched, so their metrics reflect the real class distribution.

In [ ]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

class NoduleDataset(Dataset):
    """Returns (image, label, cache_index). The cache index is carried through so out-of-fold
    predictions can be written back to the right row, and so Task 6 can join per-sample
    uncertainty to nodule metadata (diameter, reader disagreement, ...)."""

    def __init__(self, patches, labels, indices, train=False, seed=0):
        self.patches, self.labels, self.indices = patches, labels, indices
        self.train = train
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.indices)

    def _augment(self, x):
        if self.rng.random() < 0.5: x = np.flip(x, axis=2)
        if self.rng.random() < 0.5: x = np.flip(x, axis=1)
        x = np.ascontiguousarray(np.rot90(x, k=int(self.rng.integers(0, 4)), axes=(1, 2)))
        if self.rng.random() < 0.3:
            x = np.clip(x * self.rng.uniform(0.93, 1.07) +
                        self.rng.uniform(-0.04, 0.04), 0.0, 1.0)
        return x

    def __getitem__(self, i):
        j = self.indices[i]
        x = self.patches[j].astype(np.float32) / 255.0          # (K, S, S)
        if self.train:
            x = self._augment(x)
        x = torch.from_numpy(np.ascontiguousarray(x))
        if x.shape[0] == 1:
            x = x.repeat(3, 1, 1)                               # 2D baseline -> 3-channel stem
        elif x.shape[0] > 3:
            mid = x.shape[0] // 2
            x = x[mid - 1: mid + 2]
        elif x.shape[0] == 2:
            x = torch.cat([x, x[-1:]], dim=0)
        x = (x - IMAGENET_MEAN) / IMAGENET_STD
        return x, torch.tensor(int(self.labels[j])), int(j)


def make_fold_loaders(fold, cfg):
    """Train/val/test DataLoaders for one CV fold. Only the training loader augments."""
    seed = cfg["SEED"] + fold["fold"]
    tr = NoduleDataset(PATCHES, LABELS, fold["train"], train=True,  seed=seed)
    va = NoduleDataset(PATCHES, LABELS, fold["val"],   train=False, seed=seed)
    te = NoduleDataset(PATCHES, LABELS, fold["test"],  train=False, seed=seed)

    if cfg["CLASS_WEIGHTS"]:
        cnt = np.bincount(LABELS[fold["train"]], minlength=2).astype(np.float64)
        w = (1.0 / np.maximum(cnt, 1))[LABELS[fold["train"]]]
        sampler, shuffle = WeightedRandomSampler(
            torch.as_tensor(w, dtype=torch.double), len(w), replacement=True), False
    else:
        sampler, shuffle = None, True

    common = dict(num_workers=cfg["NUM_WORKERS"], pin_memory=torch.cuda.is_available())
    return (DataLoader(tr, batch_size=cfg["BATCH_SIZE"], sampler=sampler, shuffle=shuffle,
                       drop_last=len(tr) > cfg["BATCH_SIZE"], **common),
            DataLoader(va, batch_size=cfg["BATCH_SIZE"] * 2, shuffle=False, **common),
            DataLoader(te, batch_size=cfg["BATCH_SIZE"] * 2, shuffle=False, **common))


_tl, _vl, _sl = make_fold_loaders(FOLDS[0], CFG)
print("fold 0 batches -> train", len(_tl), "| val", len(_vl), "| test", len(_sl))
_xb, _yb, _jb = next(iter(_tl))
print("batch tensor:", tuple(_xb.shape), _xb.dtype, "| labels:", _yb.tolist()[:12])
del _tl, _vl, _sl

In [ ]:
# Visual sanity check, using fold 0's training set (any fold would do).
# Visual sanity check. LOOK AT THIS before trusting any metric: the nodule must sit in the middle
# of every patch. If it does not, the centroid/spacing handling in 4.1.3 is wrong and every number
# downstream is noise.
n_show = 8
fig, axes = plt.subplots(2, n_show, figsize=(1.9 * n_show, 4.6))
for row, cls in enumerate([0, 1]):
    pool = FOLDS[0]["train"][LABELS[FOLDS[0]["train"]] == cls][:n_show]
    for c in range(n_show):
        ax = axes[row, c]; ax.axis("off")
        if c >= len(pool):
            continue
        j = pool[c]
        ax.imshow(PATCHES[j][PATCHES[j].shape[0] // 2], cmap="gray", vmin=0, vmax=255)
        ax.set_title(f"{CLASS_NAMES[cls]}\nmal={MALIGNANCY[j]:.1f}", fontsize=8)
fig.suptitle(f"Centre slice of training patches  |  {CFG['PATCH_MM']:.0f} mm field of view"
             + ("   [SMOKE / PHANTOM DATA]" if IS_SMOKE else ""), fontsize=11)
plt.tight_layout(); plt.savefig(f"{CFG['OUT_DIR']}/data_samples.png", dpi=110); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
vals, cnts = np.unique(MALIGNANCY, return_counts=True)
ax[0].bar([f"{v:.1f}" for v in vals], cnts, color="#6b8fd4")
ax[0].set_title("Median malignancy rating"
                + (" (score 3 excluded)" if CFG["DROP_SCORE_3"] else ""))
ax[0].set_xlabel("median rating across readers")
ax[1].bar(CLASS_NAMES, [int((LABELS == 0).sum()), int((LABELS == 1).sum())],
          color=["#5aa469", "#c95c5c"])
ax[1].set_title("Binary class balance")
plt.tight_layout(); plt.savefig(f"{CFG['OUT_DIR']}/label_distribution.png", dpi=110); plt.show()

## 4.2 &nbsp;Models

One factory, two architectures, one shared contract. Both return raw logits of shape `(B, 2)` and
both expose their dropout modules to `enable_mc_dropout()` in Task 6, so nothing downstream needs
to know which architecture it is holding.

### Why dropout is injected at *training* time, in Task 4

MC Dropout is not a post-hoc wrapper. It approximates a Bayesian posterior only if dropout was
**active during training**, so the network learned to be robust to its own units being masked. Adding
dropout to an already-trained deterministic network and sampling from it produces noise, not
calibrated uncertainty. So `DROPOUT_P` is set now, in the Task 4 baseline, even though the
uncertainty machinery is not used until Task 6 - the alternative is retraining everything next week.

`DROPOUT_MODE`:
- `head` - dropout only before the classifier. Cheap, but the uncertainty is shallow: only the last
  layer varies, so predictive variance is systematically under-dispersed.
- `head_and_stages` - additional 2D dropout after each residual stage. This is the configuration
  the UQ literature actually validates, and the default here.

In [ ]:
import torchvision

class MCDropout2d(nn.Dropout2d):
    """Marker subclass, so the layers WE inserted can be counted and reported separately from
    whatever dropout a third-party backbone brings with it. Task 6 samples over all of them;
    the distinction is for diagnostics, not for gating."""
    pass

class MCDropout(nn.Dropout):
    pass


class ResNet18Nodule(nn.Module):
    """TASK 4 baseline: ImageNet-pretrained ResNet-18 with MC-Dropout capability."""

    def __init__(self, n_classes=2, pretrained=True, p=0.3, mode="head_and_stages"):
        super().__init__()
        weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        net = torchvision.models.resnet18(weights=weights)
        self.n_features = net.fc.in_features                     # 512

        self.stem = nn.Sequential(net.conv1, net.bn1, net.relu, net.maxpool)
        self.layer1, self.layer2 = net.layer1, net.layer2
        self.layer3, self.layer4 = net.layer3, net.layer4
        self.pool = nn.AdaptiveAvgPool2d(1)

        use_stage_drop = (mode == "head_and_stages")
        # Rising rate with depth: early features are generic and shared, late features are
        # task-specific and are where epistemic uncertainty actually lives.
        self.d1 = MCDropout2d(p * 0.25) if use_stage_drop else nn.Identity()
        self.d2 = MCDropout2d(p * 0.50) if use_stage_drop else nn.Identity()
        self.d3 = MCDropout2d(p * 0.75) if use_stage_drop else nn.Identity()
        self.d4 = MCDropout2d(p)        if use_stage_drop else nn.Identity()
        self.head_drop = MCDropout(p)
        self.fc = nn.Linear(self.n_features, n_classes)

    def forward_features(self, x):
        x = self.stem(x)
        x = self.d1(self.layer1(x))
        x = self.d2(self.layer2(x))
        x = self.d3(self.layer3(x))
        x = self.d4(self.layer4(x))
        return x

    def forward(self, x):
        f = self.pool(self.forward_features(x)).flatten(1)
        return self.fc(self.head_drop(f))


class ViTNodule(nn.Module):
    """TASK 5 comparison: ImageNet-pretrained ViT / DeiT via timm.

    CAREFUL - drop_rate is not enough. In timm 1.x, drop_rate wires ONLY the classifier-head
    dropout; every dropout inside the transformer blocks stays at p=0.0. Passing drop_rate alone
    therefore builds a model with ~50 dropout modules that are all inert, and Task 6 would then
    sample over the final linear layer alone, producing a confident-looking uncertainty estimate
    with almost no variance in it. proj_drop_rate is the argument that actually reaches
    attn.proj_drop and mlp.drop1/drop2 in every block. Verified on timm 1.0.24: 37 active
    dropout layers with it, 1 without.

    attn_drop stays at 0 deliberately: dropping attention weights perturbs the attention
    distribution itself, which is not the perturbation MC Dropout theory assumes.
    """

    def __init__(self, n_classes=2, pretrained=True, p=0.3,
                 backbone="vit_small_patch16_224", img_size=224):
        super().__init__()
        self.backbone_name = backbone
        kwargs = dict(pretrained=pretrained, num_classes=0,   # num_classes=0 -> pooled features
                      drop_rate=p * 0.5, attn_drop_rate=0.0, img_size=img_size)
        try:
            self.body = timm.create_model(backbone, proj_drop_rate=p * 0.5, **kwargs)
        except TypeError:
            # older timm without proj_drop_rate - set the block dropout rates by hand
            self.body = timm.create_model(backbone, **kwargs)
            for mod_name, mod in self.body.named_modules():
                if isinstance(mod, nn.Dropout) and ("mlp.drop" in mod_name
                                                    or "proj_drop" in mod_name):
                    mod.p = p * 0.5

        self.n_features = self.body.num_features
        self.head_drop = MCDropout(p)
        self.fc = nn.Linear(self.n_features, n_classes)

    def forward_features(self, x):
        return self.body(x)

    def forward(self, x):
        return self.fc(self.head_drop(self.body(x)))


MODEL_REGISTRY = {
    # TASK 4
    "resnet18":   lambda cfg: ResNet18Nodule(2, cfg["PRETRAINED"], cfg["DROPOUT_P"],
                                             cfg["DROPOUT_MODE"]),
    # TASK 5 - vit_tiny is the specified arm; the others are here for a sensitivity check only
    "vit_tiny":   lambda cfg: ViTNodule(2, cfg["PRETRAINED"], cfg["DROPOUT_P"],
                                        "vit_tiny_patch16_224", cfg["IMG_SIZE"]),
    "vit_small":  lambda cfg: ViTNodule(2, cfg["PRETRAINED"], cfg["DROPOUT_P"],
                                        "vit_small_patch16_224", cfg["IMG_SIZE"]),
    "deit_tiny":  lambda cfg: ViTNodule(2, cfg["PRETRAINED"], cfg["DROPOUT_P"],
                                        "deit_tiny_patch16_224", cfg["IMG_SIZE"]),
}

def build_model(name, cfg):
    if name not in MODEL_REGISTRY:
        raise KeyError(f"Unknown model '{name}'. Available: {list(MODEL_REGISTRY)}")
    return MODEL_REGISTRY[name](cfg).to(DEVICE)


def count_active_dropout(model):
    """Dropout modules with p > 0. A module with p == 0 is inert - it injects no noise - so
    counting every Dropout instance would hide exactly the timm bug described above."""
    return sum(1 for m in model.modules()
               if isinstance(m, (nn.Dropout, nn.Dropout1d, nn.Dropout2d, nn.Dropout3d,
                                 nn.AlphaDropout)) and m.p > 0)


for name in MODEL_REGISTRY:
    try:
        m = MODEL_REGISTRY[name](dict(CFG, PRETRAINED=False))
        n_par   = sum(q.numel() for q in m.parameters()) / 1e6
        n_act   = count_active_dropout(m)
        n_inert = sum(1 for mod in m.modules() if isinstance(mod, nn.Dropout) and mod.p == 0)
        with torch.no_grad():
            out = m(torch.zeros(2, 3, CFG["IMG_SIZE"], CFG["IMG_SIZE"]))
        flag = "" if n_act >= 3 else "   <-- TOO FEW for meaningful MC Dropout, see Task 6"
        print(f"{name:12s} {n_par:6.1f}M params | {n_act:3d} ACTIVE dropout "
              f"({n_inert} inert) | out {tuple(out.shape)}{flag}")
        del m
    except Exception as e:
        print(f"{name:12s} FAILED: {type(e).__name__}: {e}")

## 4.3 &nbsp;Evaluation harness

Everything in this section is architecture-agnostic and gets reused by Tasks 4, 5 and 6 unchanged.
It implements the specific metrics the revised proposal (Section 6, "Statistical Evaluation" and
"False-Negative Evaluation") commits to:

| Metric | Why the proposal needs it |
|---|---|
| Accuracy / precision / recall / F1 / AUROC / AUPRC | baseline comparison table |
| **Sensitivity at fixed specificity** 0.90 / 0.95 / 0.99 | the false-negative requirement; a single accuracy number hides FNs |
| **Bootstrap 95% CIs**, resampled **by patient** | an interval, not a point estimate; resampling nodules would give falsely narrow CIs |
| **ECE + Brier** | calibration - required before any confidence threshold means anything |
| **DeLong test** | AUROC comparison ResNet vs ViT with a p-value |
| **McNemar test** | paired error comparison at the operating point |

The bootstrap resamples **patients, not nodules**. This matters: nodules from one patient are
correlated, so nodule-level resampling understates the variance and produces CIs that are too
narrow to be honest.

In [ ]:
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                             accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, brier_score_loss)
from scipy import stats


def expected_calibration_error(y_true, p_pos, n_bins=15):
    """Equal-width binning ECE. Reported alongside Brier because Brier conflates calibration
    with discrimination, while ECE isolates the calibration part."""
    conf = np.where(p_pos >= 0.5, p_pos, 1.0 - p_pos)      # confidence in the predicted class
    correct = ((p_pos >= 0.5).astype(int) == y_true).astype(float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece, per_bin = 0.0, []
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        m = (conf > lo) & (conf <= hi) if i > 0 else (conf >= lo) & (conf <= hi)
        if not m.any():
            per_bin.append((0.5 * (lo + hi), np.nan, np.nan, 0)); continue
        acc, avg_conf = correct[m].mean(), conf[m].mean()
        ece += (m.sum() / len(conf)) * abs(acc - avg_conf)
        per_bin.append((0.5 * (lo + hi), acc, avg_conf, int(m.sum())))
    return float(ece), per_bin


def sensitivity_at_specificity(y_true, p_pos, target_spec):
    """Highest achievable sensitivity subject to specificity >= target, plus the threshold.
    This is the clinical operating point: in screening you fix the false-positive burden you
    can tolerate and then ask how many cancers you catch."""
    fpr, tpr, thr = roc_curve(y_true, p_pos)
    spec = 1.0 - fpr
    ok = spec >= target_spec
    if not ok.any():
        return np.nan, np.nan, np.nan
    i = int(np.argmax(np.where(ok, tpr, -np.inf)))
    return float(tpr[i]), float(spec[i]), float(thr[i])


def core_metrics(y_true, p_pos, threshold=0.5, cfg=CFG):
    y_pred = (p_pos >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    ece, _ = expected_calibration_error(y_true, p_pos, cfg["N_ECE_BINS"])

    out = dict(
        threshold   = float(threshold),
        n           = int(len(y_true)),
        accuracy    = float(accuracy_score(y_true, y_pred)),
        precision   = float(precision_score(y_true, y_pred, zero_division=0)),
        recall      = float(recall_score(y_true, y_pred, zero_division=0)),
        specificity = float(tn / (tn + fp)) if (tn + fp) else np.nan,
        f1          = float(f1_score(y_true, y_pred, zero_division=0)),
        auroc       = float(roc_auc_score(y_true, p_pos)) if len(np.unique(y_true)) > 1 else np.nan,
        auprc       = float(average_precision_score(y_true, p_pos)),
        brier       = float(brier_score_loss(y_true, p_pos)),
        ece         = float(ece),
        tp=int(tp), fp=int(fp), tn=int(tn), fn=int(fn),
        false_negative_rate = float(fn / (fn + tp)) if (fn + tp) else np.nan,
    )
    for ts in cfg["TARGET_SPECIFICITIES"]:
        sens, spec, thr = sensitivity_at_specificity(y_true, p_pos, ts)
        out[f"sens_at_spec_{int(ts*100)}"] = sens
        out[f"thr_at_spec_{int(ts*100)}"]  = thr
    return out


def bootstrap_ci(y_true, p_pos, groups, metric_fn, n_boot=1000, seed=42, alpha=0.05):
    """Patient-level (cluster) bootstrap: resample PATIENTS with replacement, take all of a
    resampled patient's nodules. Nodule-level resampling would ignore within-patient
    correlation and report intervals that are too narrow."""
    rng = np.random.default_rng(seed)
    uniq = np.unique(groups)
    by_group = {g: np.where(groups == g)[0] for g in uniq}
    vals = []
    for _ in range(n_boot):
        pick = rng.choice(uniq, size=len(uniq), replace=True)
        idx = np.concatenate([by_group[g] for g in pick])
        if len(np.unique(y_true[idx])) < 2:
            continue
        try:
            vals.append(metric_fn(y_true[idx], p_pos[idx]))
        except Exception:
            continue
    if not vals:
        return np.nan, np.nan, np.nan
    vals = np.asarray(vals, dtype=float)
    return (float(np.mean(vals)),
            float(np.percentile(vals, 100 * alpha / 2)),
            float(np.percentile(vals, 100 * (1 - alpha / 2))))

In [ ]:
# ---- DeLong test: is the AUROC difference between two models statistically significant? ----
# Fast implementation of Sun & Xu (2014), the standard for correlated ROC curves (same test set,
# two models -> the two AUCs are correlated, so an unpaired test would be wrong).

def _compute_midrank(x):
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N, dtype=float)
    T2[J] = T
    return T2


def _fast_delong(preds_sorted, n_pos):
    """preds_sorted: (k_models, n_samples) with all positives first."""
    m, k = n_pos, preds_sorted.shape[0]
    n = preds_sorted.shape[1] - m
    pos, neg = preds_sorted[:, :m], preds_sorted[:, m:]

    tx = np.empty([k, m]); ty = np.empty([k, n]); tz = np.empty([k, m + n])
    for r in range(k):
        tx[r] = _compute_midrank(pos[r])
        ty[r] = _compute_midrank(neg[r])
        tz[r] = _compute_midrank(preds_sorted[r])

    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    sx = np.atleast_2d(np.cov(v01)); sy = np.atleast_2d(np.cov(v10))
    return aucs, sx / m + sy / n


def delong_roc_test(y_true, p_a, p_b):
    """Two-sided DeLong test. Returns (auc_a, auc_b, z, p_value)."""
    y_true = np.asarray(y_true).astype(int)
    order = np.argsort(-y_true, kind="mergesort")      # positives first, ties stable
    n_pos = int(y_true.sum())
    preds = np.vstack([np.asarray(p_a, float), np.asarray(p_b, float)])[:, order]

    aucs, cov = _fast_delong(preds, n_pos)
    l = np.array([[1.0, -1.0]])
    var = float(l @ cov @ l.T)
    if var <= 0:
        return float(aucs[0]), float(aucs[1]), np.nan, np.nan
    z = float((aucs[0] - aucs[1]) / np.sqrt(var))
    # norm.sf, not 1 - norm.cdf: the latter underflows to exactly 0.0 for |z| > ~8 and would
    # print "p = 0", which is not a reportable p-value.
    return float(aucs[0]), float(aucs[1]), z, float(2.0 * stats.norm.sf(abs(z)))


def mcnemar_test(y_true, pred_a, pred_b):
    """Exact McNemar on the discordant pairs: do the two models make DIFFERENT mistakes on the
    same cases? Complements DeLong, which compares ranking rather than decisions."""
    a_ok = (np.asarray(pred_a) == y_true)
    b_ok = (np.asarray(pred_b) == y_true)
    n01 = int((~a_ok & b_ok).sum())      # A wrong, B right
    n10 = int((a_ok & ~b_ok).sum())      # A right, B wrong
    if n01 + n10 == 0:
        return n01, n10, 1.0
    p = float(stats.binomtest(min(n01, n10), n01 + n10, 0.5, alternative="two-sided").pvalue)
    return n01, n10, p


# Self-check on synthetic data: a clearly better scorer must come out with a small p-value.
_rng = np.random.default_rng(0)
_y = np.r_[np.ones(120), np.zeros(120)].astype(int)
_good = np.r_[_rng.normal(1.6, 1, 120), _rng.normal(0, 1, 120)]
_weak = np.r_[_rng.normal(0.4, 1, 120), _rng.normal(0, 1, 120)]
_a, _b, _z, _p = delong_roc_test(_y, _good, _weak)
print(f"DeLong self-check: auc_good={_a:.3f} auc_weak={_b:.3f} z={_z:.2f} p={_p:.2e}")
assert _p < 0.01 and _a > _b, "DeLong implementation is broken"
# A large |z| must still give a representable p. 1 - norm.cdf(|z|) hits exactly 0.0 around
# |z| = 8.3, so a strong result would print "p = 0" - not a reportable number.
_p_extreme = float(2.0 * stats.norm.sf(12.0))
assert _p_extreme > 0.0, "p-value underflowed - delong_roc_test must use norm.sf, not 1-norm.cdf"
print(f"  p at z=12 stays representable: {_p_extreme:.3e}  "
      f"(1-norm.cdf would give {2.0*(1.0-stats.norm.cdf(12.0)):.1f})")
print("McNemar self-check:", mcnemar_test(_y, (_good > 0.8).astype(int), (_weak > 0.2).astype(int)))

## 4.4 &nbsp;Training loop

One fold's worth of training. Called `N_FOLDS` times by Section 4.5.

Model selection is on **validation AUROC**, not validation accuracy. With class imbalance and a
threshold that gets re-tuned for the clinical operating point anyway, accuracy at 0.5 is the wrong
quantity to select on. Validation here is always a patient-disjoint slice of the *training* pool, so
the fold's test set stays untouched until the model is final.

Both notebooks use this identical loop, optimiser, schedule, early-stopping rule and seed. The only
difference between Task 4 and Task 5 is `CFG["MODEL"]`. Any other difference and a reviewer can
argue the comparison measures tuning effort rather than architecture.

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

@torch.no_grad()
def predict_deterministic(model, loader):
    """Standard single-pass inference. Dropout OFF - this is the Task 4/5 baseline prediction."""
    model.eval()
    P, Y, J = [], [], []
    for x, y, j in loader:
        logits = model(x.to(DEVICE, non_blocking=True))
        P.append(torch.softmax(logits.float(), dim=1)[:, 1].cpu().numpy())
        Y.append(y.numpy()); J.append(np.asarray(j))
    return np.concatenate(P), np.concatenate(Y), np.concatenate(J)


def train_one_fold(name, cfg, fold, train_loader, val_loader, verbose=True):
    """Train on one fold. Returns (model, history, checkpoint_path)."""
    set_seed(cfg["SEED"] + fold["fold"])
    model = build_model(name, cfg)

    if cfg["CLASS_WEIGHTS"]:
        cnt = np.bincount(LABELS[fold["train"]], minlength=2).astype(np.float32)
        # The sampler already balances the stream; this mild residual weight guards the case
        # where CLASS_WEIGHTS sampling is turned off.
        w = torch.tensor(cnt.sum() / (2.0 * np.maximum(cnt, 1)), dtype=torch.float32, device=DEVICE)
        criterion = nn.CrossEntropyLoss(weight=w)
    else:
        criterion = nn.CrossEntropyLoss()

    opt = AdamW(model.parameters(), lr=cfg["LR"], weight_decay=cfg["WEIGHT_DECAY"])
    sched = CosineAnnealingLR(opt, T_max=cfg["EPOCHS"])
    use_amp = cfg["AMP"] and DEVICE.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    ckpt = f"{cfg['OUT_DIR']}/{name}_fold{fold['fold']}_best.pt"
    best_auc, best_epoch, bad, history = -np.inf, -1, 0, []
    t0 = time.time()

    for epoch in range(1, cfg["EPOCHS"] + 1):
        model.train()
        tot, seen = 0.0, 0
        for x, y, _ in train_loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=use_amp):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            tot += loss.item() * x.size(0); seen += x.size(0)
        sched.step()

        p_val, y_val, _ = predict_deterministic(model, val_loader)
        val_auc = roc_auc_score(y_val, p_val) if len(np.unique(y_val)) > 1 else np.nan
        val_acc = accuracy_score(y_val, (p_val >= 0.5).astype(int))
        history.append(dict(fold=fold["fold"], epoch=epoch, train_loss=tot / max(seen, 1),
                            val_auroc=float(val_auc), val_acc=float(val_acc),
                            lr=opt.param_groups[0]["lr"]))

        improved = np.isfinite(val_auc) and val_auc > best_auc + 1e-5
        if improved:
            best_auc, best_epoch, bad = float(val_auc), epoch, 0
            torch.save(dict(model=model.state_dict(), cfg=cfg, name=name,
                            fold=fold["fold"], epoch=epoch, val_auroc=best_auc,
                            fingerprint=FINGERPRINT), ckpt)
        else:
            bad += 1

        if verbose:
            print(f"    epoch {epoch:3d}/{cfg['EPOCHS']} | loss {history[-1]['train_loss']:.4f} "
                  f"| val AUROC {val_auc:.4f} | val acc {val_acc:.4f}"
                  f"{'  <- best' if improved else ''}")
        if bad >= cfg["PATIENCE"]:
            print(f"    early stop at epoch {epoch} (no val AUROC gain for {bad} epochs)")
            break

    model.load_state_dict(torch.load(ckpt, map_location=DEVICE)["model"])
    print(f"    restored best epoch {best_epoch} (val AUROC {best_auc:.4f}) "
          f"| {(time.time()-t0)/60:.1f} min")
    return model, pd.DataFrame(history), ckpt

In [ ]:
def evaluate_and_report(name, p_test, y_test, j_test, cfg=CFG, tag="deterministic"):
    """Full metric block + bootstrap CIs + saved artefacts. Used by Tasks 4, 5 and 6."""
    groups = PATIENT_ID[j_test]
    m = core_metrics(y_test, p_test, 0.5, cfg)
    m.update(model=name, tag=tag, smoke=IS_SMOKE,
             n_test_patients=int(len(np.unique(groups))))

    for key, fn in [("auroc", lambda a, b: roc_auc_score(a, b)),
                    ("auprc", lambda a, b: average_precision_score(a, b)),
                    ("accuracy", lambda a, b: accuracy_score(a, (b >= 0.5).astype(int))),
                    ("recall", lambda a, b: recall_score(a, (b >= 0.5).astype(int),
                                                         zero_division=0)),
                    ("f1", lambda a, b: f1_score(a, (b >= 0.5).astype(int), zero_division=0))]:
        mean, lo, hi = bootstrap_ci(y_test, p_test, groups, fn,
                                    cfg["N_BOOTSTRAP"], cfg["SEED"])
        m[f"{key}_ci_low"], m[f"{key}_ci_high"] = lo, hi

    banner = f"  {name} / {tag}  " + ("[SMOKE - NOT REPORTABLE]" if IS_SMOKE else "")
    print("=" * 78); print(banner); print("=" * 78)
    print(f"  test set: {m['n']} nodules from {m['n_test_patients']} patients")
    print(f"  AUROC        {m['auroc']:.4f}  [{m['auroc_ci_low']:.4f}, {m['auroc_ci_high']:.4f}]")
    print(f"  AUPRC        {m['auprc']:.4f}  [{m['auprc_ci_low']:.4f}, {m['auprc_ci_high']:.4f}]")
    print(f"  Accuracy     {m['accuracy']:.4f}  [{m['accuracy_ci_low']:.4f}, "
          f"{m['accuracy_ci_high']:.4f}]")
    print(f"  Sensitivity  {m['recall']:.4f}  [{m['recall_ci_low']:.4f}, {m['recall_ci_high']:.4f}]")
    print(f"  Specificity  {m['specificity']:.4f}")
    print(f"  Precision    {m['precision']:.4f}      F1  {m['f1']:.4f}")
    print(f"  Brier        {m['brier']:.4f}      ECE {m['ece']:.4f}")
    print(f"  Confusion    TP={m['tp']} FP={m['fp']} TN={m['tn']} FN={m['fn']}"
          f"   (FN rate {m['false_negative_rate']:.3f})")
    print("  -- clinical operating points (the false-negative requirement) --")
    for ts in cfg["TARGET_SPECIFICITIES"]:
        s = m[f"sens_at_spec_{int(ts*100)}"]
        t = m[f"thr_at_spec_{int(ts*100)}"]
        print(f"    sensitivity @ specificity>={ts:.2f} : "
              + (f"{s:.4f}  (threshold {t:.4f})" if np.isfinite(s) else "not achievable"))

    np.savez(f"{cfg['OUT_DIR']}/{name}_{tag}_predictions.npz",
             p=p_test, y=y_test, cache_index=j_test, patient_id=groups)
    with open(f"{cfg['OUT_DIR']}/{name}_{tag}_metrics.json", "w") as f:
        json.dump(m, f, indent=2, default=float)
    return m


def plot_roc_pr(curves, title):
    """curves: list of (label, y_true, p_pos)."""
    from sklearn.metrics import precision_recall_curve
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
    for label, y, p in curves:
        fpr, tpr, _ = roc_curve(y, p)
        ax[0].plot(fpr, tpr, lw=1.8, label=f"{label} (AUC {roc_auc_score(y, p):.3f})")
        pr, rc, _ = precision_recall_curve(y, p)
        ax[1].plot(rc, pr, lw=1.8, label=f"{label} (AP {average_precision_score(y, p):.3f})")
    ax[0].plot([0, 1], [0, 1], "k:", lw=1)
    for ts in CFG["TARGET_SPECIFICITIES"]:
        ax[0].axvline(1 - ts, color="grey", ls="--", lw=0.7)
        ax[0].text(1 - ts, 0.02, f"spec {ts:.2f}", rotation=90, fontsize=7, color="grey")
    ax[0].set_xlabel("1 - specificity"); ax[0].set_ylabel("sensitivity")
    ax[0].set_title("ROC"); ax[0].legend(loc="lower right", fontsize=8)
    ax[1].set_xlabel("recall"); ax[1].set_ylabel("precision")
    ax[1].set_title("Precision-Recall"); ax[1].legend(loc="lower left", fontsize=8)
    fig.suptitle(title + ("   [SMOKE]" if IS_SMOKE else ""))
    plt.tight_layout()
    plt.savefig(f"{CFG['OUT_DIR']}/{title.replace(' ','_').replace('/','-')}_roc_pr.png", dpi=110)
    plt.show()


RESULTS = []   # accumulates every metric dict produced in this notebook

---

## 6.0 &nbsp;What MC Dropout gives you, and what it does not

Keep dropout active at inference and run `T` forward passes (`MC_PASSES`, default 50). Each pass is
a draw from an approximate posterior over weights, so instead of one probability per nodule you get
a distribution over `T` of them. Three quantities come out of that spread, and they are **not**
interchangeable:

| Quantity | Formula | Reads as |
|---|---|---|
| **Predictive entropy** | `H(mean_t p_t)` | total uncertainty |
| **Expected entropy** | `mean_t H(p_t)` | **aleatoric** - ambiguity in the image itself; more data will not fix it |
| **Mutual information** | `H(mean) - mean H` | **epistemic** - the model does not know; more data *would* fix it |

All three are normalised by `ln 2`, so each lands in [0, 1] for this binary task.

The split is what makes the signal useful for routing. High **epistemic** uncertainty says the case
falls outside what this model learned, so a different model, test-time augmentation or an ensemble
may help. High **aleatoric** uncertainty on a genuinely ambiguous lesion means extra compute is
wasted and the case belongs with a radiologist. A single "confidence" scalar cannot tell those two
apart.

### Two implementation details that quietly ruin this

**1. `model.train()` is the wrong switch.** It also puts **BatchNorm** into training mode, which
recomputes batch statistics from the inference batch. Predictions would then depend on which other
nodules happen to share the batch — non-determinism that has nothing to do with model uncertainty.
`enable_mc_dropout()` flips **only** dropout layers and leaves normalisation in eval mode.

**2. Counting dropout layers is not enough; they must have `p > 0`.** A `Dropout` module with
`p = 0.0` sits in the graph and injects nothing. `timm` builds ViTs with ~50 such inert modules
unless `proj_drop_rate` is set, so an existence check would pass on a model whose `T` passes are all
identical. `enable_mc_dropout()` counts only active layers and refuses to run below a minimum, and
`mc_dropout_predict()` reports the mean spread across passes so a degenerate run is visible.

### Why this must run per fold

Task 4 produced **five** models, one per CV fold. Every nodule has to be scored by the fold model
that never saw its patient — the same out-of-fold discipline as Task 4. Running one model over all
nodules would measure uncertainty on data it trained on, and trained-on data always looks
confident. So the loop below is per fold, and the checkpoint's stored fingerprint is checked against
this run's before anything is computed: if the fold assignment differs, out-of-fold is not
out-of-fold any more.

In [ ]:
DROPOUT_TYPES = (nn.Dropout, nn.Dropout1d, nn.Dropout2d, nn.Dropout3d, nn.AlphaDropout)

def enable_mc_dropout(model, min_active=3):
    """Activate dropout for sampling while keeping BatchNorm/LayerNorm in eval mode.

    Counts only layers with p > 0 - an inert Dropout(p=0) contributes nothing, so an existence
    check would happily pass on a model whose T passes are all identical.
    """
    model.eval()                                    # everything deterministic first...
    active = [m for m in model.modules() if isinstance(m, DROPOUT_TYPES) and m.p > 0]
    for m in active:                                # ...then re-enable real dropout only
        m.train()

    if len(active) < min_active:
        raise RuntimeError(
            f"Only {len(active)} dropout layer(s) with p > 0 - MC Dropout would return nearly "
            f"identical passes and its 'uncertainty' would be an artefact. Check CFG['DROPOUT_P'] "
            f"and DROPOUT_MODE in the run that produced this checkpoint, then RETRAIN: dropout "
            f"must be active during TRAINING, not bolted on afterwards.")
    return len(active)


@torch.no_grad()
def mc_dropout_predict(model, loader, n_passes, seed=0, label=""):
    """T stochastic forward passes -> (T, N, C) probabilities, plus labels and cache indices.

    Kept unreduced so any uncertainty measure can be derived afterwards without re-running the
    passes, which is the expensive part.
    """
    n_active = enable_mc_dropout(model)
    torch.manual_seed(seed)

    all_passes, Y, J = [], None, None
    t0 = time.time()
    for t in range(n_passes):
        P, ys, js = [], [], []
        for x, y, j in loader:
            logits = model(x.to(DEVICE, non_blocking=True))
            P.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
            if t == 0:
                ys.append(y.numpy()); js.append(np.asarray(j))
        all_passes.append(np.concatenate(P, axis=0))
        if t == 0:
            Y, J = np.concatenate(ys), np.concatenate(js)

    model.eval()
    probs = np.stack(all_passes, axis=0)             # (T, N, C)
    spread = float(probs[:, :, 1].std(axis=0).mean())
    print(f"      MC {label}: {n_passes} passes x {probs.shape[1]} nodules | "
          f"{n_active} active dropout | mean spread {spread:.5f} | {time.time()-t0:.1f}s")
    if spread < 1e-3:
        print("      WARNING: passes are nearly identical - the uncertainty below is not "
              "meaningful.")
    return probs, Y, J


UNC_KEYS = ["mean_prob_positive", "confidence", "predictive_entropy", "aleatoric",
            "epistemic", "std_positive", "vote_disagreement"]

def uncertainty_from_passes(probs, eps=1e-12):
    """(T, N, C) -> per-sample uncertainty decomposition, entropies normalised to [0, 1]."""
    mean_p = probs.mean(axis=0)                                        # (N, C)
    pred_entropy = -(mean_p * np.log(mean_p + eps)).sum(axis=1)        # total
    exp_entropy  = -(probs * np.log(probs + eps)).sum(axis=2).mean(0)  # aleatoric
    mutual_info  = pred_entropy - exp_entropy                          # epistemic
    ln2 = np.log(2.0)
    votes = (probs[:, :, 1] >= 0.5).mean(axis=0)
    return dict(
        mean_prob_positive = mean_p[:, 1],
        confidence         = mean_p.max(axis=1),
        predictive_entropy = pred_entropy / ln2,
        aleatoric          = exp_entropy / ln2,
        epistemic          = np.clip(mutual_info, 0.0, None) / ln2,
        std_positive       = probs[:, :, 1].std(axis=0),
        # spread of the T hard decisions - 0 = all passes agree, 1 = even split
        vote_disagreement  = np.minimum(votes, 1.0 - votes) * 2.0,
    )

## 6.1 &nbsp;Locating Task 4's fold checkpoints

MC Dropout samples the trained weights, so it needs Task 4's five fold checkpoints -
`<model>_fold<k>_best.pt`. There are two ways to supply them.

**Preferred - attach them.** Add the Task 4 run's output to this notebook:
`Add Input` -> `Your Work` / `Notebook Output` -> pick the saved version of the Task 4 notebook.
Kaggle keeps everything that was in `/kaggle/working` for any version you saved, and the five
checkpoints are ~215 MB in total, well inside the limit. This is the right path because it samples
*exactly* the models whose numbers you already reported.

**Fallback - retrain here.** If the checkpoints are gone (an interactive session that was never
saved as a version), set `CFG["RETRAIN_IF_MISSING"] = True`. Task 6 then trains the five folds
itself before sampling them. At this dataset size that is roughly 51 min on CPU, and the protocol
is reproduced exactly, because the folds are derived from the cache and `SEED` rather than stored
in the checkpoints.

The one caveat with retraining: fresh models are *not* bit-identical to Task 4's, so the
deterministic column will land near, but not exactly on, the AUROC you reported. The cell prints
both so you can see the gap. If it is small, the uncertainty analysis is sound; if it is large,
something differs between the runs and is worth finding before reporting anything.

### Why the fingerprint is checked

The fingerprint covers the fold assignment, so a mismatch means the checkpoints were trained on a
different partition - and the nodules this notebook treats as "held out" would include ones those
models trained on. Uncertainty measured that way is meaninglessly low, and nothing downstream would
look wrong. So it is checked before any GPU time is spent, and a mismatch stops the run.

In [ ]:
def find_fold_checkpoints(model_name, cfg):
    """Locate one checkpoint per fold, from this session's output or any attached dataset."""
    found = {}
    for root in [cfg["OUT_DIR"], cfg["WORK"], "/kaggle/input"]:
        for p in sorted(glob.glob(f"{root}/**/{model_name}_fold*_best.pt", recursive=True)):
            try:
                k = int(os.path.basename(p).split("_fold")[1].split("_")[0])
            except (IndexError, ValueError):
                continue
            found.setdefault(k, p)
    return found


def load_fold_model(path, cfg, expect_fingerprint=None):
    ck = torch.load(path, map_location=DEVICE, weights_only=False)
    got = ck.get("fingerprint")
    if expect_fingerprint is not None and got is not None and got != expect_fingerprint:
        raise RuntimeError(
            f"FINGERPRINT MISMATCH on {os.path.basename(path)}\n"
            f"  checkpoint was trained on : {got}\n"
            f"  this session's data is    : {expect_fingerprint}\n"
            f"The fold assignment differs, so these nodules are NOT out-of-fold for this model "
            f"and any uncertainty computed from it would be invalid.\n"
            f"Fix: attach the same cache dataset that Task 4 used (the one whose fingerprint is "
            f"{got}), or re-run Task 4 on this cache.")
    model = build_model(ck["name"], cfg)
    model.load_state_dict(ck["model"])
    model.eval()
    return model, ck


CKPTS_BY_FOLD = find_fold_checkpoints(CFG["MODEL"], CFG)
print(f"looking for {CFG['MODEL']}_fold*_best.pt")
for k in sorted(CKPTS_BY_FOLD):
    print(f"  fold {k}: {CKPTS_BY_FOLD[k]}")

missing = [k for k in range(CFG["N_FOLDS"]) if k not in CKPTS_BY_FOLD]

if missing and CFG.get("RETRAIN_IF_MISSING", False):
    print(f"\nCheckpoints missing for fold(s) {missing} - RETRAIN_IF_MISSING is on, "
          f"training them here.")
    print(f"  Roughly {max(5, round(51 * len(LABELS) / 322.0))} min expected on CPU, faster on a "
          f"working GPU.")
    print("  These will be FRESH models, not the exact ones Task 4 reported - see the note above.")
    for k in missing:
        f = FOLDS[k]
        print(f"\n--- training fold {k} ---")
        tl, vl, _sl = make_fold_loaders(f, CFG)
        _model, _hist, _ckpt = train_one_fold(CFG["MODEL"], CFG, f, tl, vl)
        CKPTS_BY_FOLD[k] = _ckpt
        del _model, tl, vl, _sl
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    missing = [k for k in range(CFG["N_FOLDS"]) if k not in CKPTS_BY_FOLD]

if missing:
    raise FileNotFoundError(
        f"Missing checkpoints for fold(s) {missing}.\n"
        f"\nOPTION 1 (preferred) - attach Task 4's output:\n"
        f"  Add Input -> Your Work / Notebook Output -> the saved version of the Task 4 notebook.\n"
        f"  It should contain {CFG['MODEL']}_fold0_best.pt ... "
        f"_fold{CFG['N_FOLDS']-1}_best.pt (~215 MB).\n"
        f"\nOPTION 2 - retrain them here:\n"
        f"  Set CFG['RETRAIN_IF_MISSING'] = True in section 4.0 and re-run. Takes about "
        f"{max(5, round(51 * len(LABELS) / 322.0))} min\n"
        f"  on CPU at this dataset size. The folds are reproduced from the cache and SEED, so the\n"
        f"  protocol is identical - only the weights are freshly trained.\n"
        f"\nIf Task 4 was only ever run interactively and never saved as a version, its\n"
        f"/kaggle/working was discarded and Option 2 is the way forward.")

# verify every checkpoint agrees with this session's data before spending any GPU time
for k in sorted(CKPTS_BY_FOLD):
    _m, _ck = load_fold_model(CKPTS_BY_FOLD[k], CFG, expect_fingerprint=FINGERPRINT)
    print(f"  fold {k}: epoch {_ck.get('epoch','?')}, val AUROC {_ck.get('val_auroc',float('nan')):.4f}"
          f", fingerprint {_ck.get('fingerprint','(none)')}  OK")
    del _m
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f"\nAll {CFG['N_FOLDS']} checkpoints present and consistent with fingerprint {FINGERPRINT}")

## 6.2 &nbsp;Out-of-fold MC Dropout

For each fold: load its checkpoint, run one deterministic pass (the Task 4 baseline prediction) and
`T` stochastic passes over that fold's held-out nodules, then write both into full-dataset vectors.
Every nodule ends up with a deterministic probability, an MC-averaged probability and its
uncertainty decomposition, all produced by a model that never saw its patient.

The fold's **validation** slice is also sampled, but only to calibrate the abstention thresholds in
6.7. Those thresholds must never be chosen on the out-of-fold set that reports the results.

> Cost: `T` &times; a forward pass, over test + validation, &times; 5 folds. At `MC_PASSES = 50` on
> a few hundred nodules this is minutes on a T4, but it scales linearly — lower `MC_PASSES` to 20
> while debugging.

In [ ]:
def mc_dropout_over_folds(model_name, cfg, folds, ckpts):
    """Per-fold MC dropout -> full-dataset OOF vectors + pooled validation for calibration."""
    N = len(LABELS)
    oof = dict(p_det=np.full(N, np.nan), fold=np.full(N, -1, dtype=np.int64))
    for k in UNC_KEYS:
        oof[k] = np.full(N, np.nan)

    val_pool, per_fold = [], []
    t0 = time.time()

    for f in folds:
        k = f["fold"]
        print(f"\n--- fold {k} ---")
        model, _ = load_fold_model(ckpts[k], cfg, expect_fingerprint=FINGERPRINT)
        tl, vl, sl = make_fold_loaders(f, cfg)

        # deterministic single pass = the Task 4 baseline behaviour
        p_det, y_t, j_t = predict_deterministic(model, sl)
        oof["p_det"][j_t] = p_det
        oof["fold"][j_t] = k

        # stochastic passes on the held-out fold
        probs, _, j_mc = mc_dropout_predict(model, sl, cfg["MC_PASSES"],
                                            cfg["SEED"] + k, label="test")
        u = uncertainty_from_passes(probs)
        for key in UNC_KEYS:
            oof[key][j_mc] = u[key]

        # validation slice, for threshold calibration only
        probs_v, y_v, j_v = mc_dropout_predict(model, vl, cfg["MC_PASSES"],
                                               cfg["SEED"] + 500 + k, label="val ")
        uv = uncertainty_from_passes(probs_v)
        val_pool.append(dict(fold=k, y=y_v, j=j_v, **{key: uv[key] for key in UNC_KEYS}))

        m_det = core_metrics(LABELS[j_t], p_det, 0.5, cfg)
        m_mc  = core_metrics(LABELS[j_mc], u["mean_prob_positive"], 0.5, cfg)
        per_fold.append(dict(fold=k,
                             auroc_det=m_det["auroc"], auroc_mc=m_mc["auroc"],
                             ece_det=m_det["ece"], ece_mc=m_mc["ece"],
                             brier_det=m_det["brier"], brier_mc=m_mc["brier"],
                             acc_det=m_det["accuracy"], acc_mc=m_mc["accuracy"],
                             fn_det=m_det["fn"], fn_mc=m_mc["fn"]))
        print(f"      fold {k}: AUROC det {m_det['auroc']:.4f} -> MC {m_mc['auroc']:.4f} | "
              f"ECE {m_det['ece']:.4f} -> {m_mc['ece']:.4f} | "
              f"Brier {m_det['brier']:.4f} -> {m_mc['brier']:.4f}")

        del model, tl, vl, sl
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    for key in ["p_det"] + UNC_KEYS:
        assert not np.isnan(oof[key]).any(), f"{key} has gaps - a fold did not cover its nodules"
    print(f"\nMC dropout complete over {len(folds)} folds in {(time.time()-t0)/60:.1f} min")

    # Pooled validation. Note: val slices come from each fold's TRAINING pool, so a nodule can
    # appear in more than one fold's validation set. That is fine for picking a threshold, but it
    # means the pooled set contains repeats and is not an unbiased performance estimate.
    VAL = {key: np.concatenate([v[key] for v in val_pool]) for key in UNC_KEYS}
    VAL["y"] = np.concatenate([v["y"] for v in val_pool])
    VAL["j"] = np.concatenate([v["j"] for v in val_pool])
    print(f"pooled validation for calibration: {len(VAL['y'])} predictions "
          f"({len(np.unique(VAL['j']))} distinct nodules, repeats across folds)")
    return oof, VAL, pd.DataFrame(per_fold)


OOF, VAL, MC_FOLD_TABLE = mc_dropout_over_folds(CFG["MODEL"], CFG, FOLDS, CKPTS_BY_FOLD)
P_DET, P_MC = OOF["p_det"], OOF["mean_prob_positive"]

# If Task 4's own out-of-fold predictions are attached, cross-check that the models sampled here
# behave like the ones whose numbers were reported. Cheap, and it catches a wrong-checkpoint mix-up
# that the fingerprint alone would not.
_t4 = None
for _p in glob.glob(f"/kaggle/input/**/{CFG['MODEL']}_oof_predictions.npz", recursive=True) + \
          glob.glob(f"{CFG['WORK']}/**/{CFG['MODEL']}_oof_predictions.npz", recursive=True):
    try:
        with np.load(_p, allow_pickle=True) as _z:
            if len(_z["p"]) == len(LABELS):
                _t4 = (np.asarray(_z["p"]), str(_z["fingerprint"]) if "fingerprint" in _z.files
                       else "?", _p)
                break
    except Exception:
        continue

if _t4 is not None:
    _p4, _fp4, _src = _t4
    _a_here = roc_auc_score(LABELS, P_DET)
    _a_t4 = roc_auc_score(LABELS, _p4)
    _corr = float(np.corrcoef(P_DET, _p4)[0, 1])
    print(f"\nCross-check against the reported Task 4 predictions ({os.path.basename(_src)}):")
    print(f"  deterministic AUROC here {_a_here:.4f}  vs reported {_a_t4:.4f}  "
          f"(diff {_a_here - _a_t4:+.4f})")
    print(f"  per-nodule correlation   {_corr:.4f}")
    print(f"  fingerprint              {_fp4}  vs this session {FINGERPRINT}")
    if abs(_a_here - _a_t4) > 0.02 or _corr < 0.95:
        print("  NOTE: these differ more than freshly-trained-but-equivalent models should. "
              "Likely retrained weights rather than Task 4's originals - fine for the uncertainty "
              "analysis, but do not present the deterministic column as Task 4's result.")
    else:
        print("  Consistent with the reported baseline.")
else:
    print("\n(Task 4's *_oof_predictions.npz not attached - skipping the cross-check. "
          "Attach it to have this verified automatically.)")

## 6.3 &nbsp;Calibration - the headline deliverable

The task asks for **ECE and Brier score**. Both are reported for the deterministic baseline and for
the MC-averaged prediction, so the effect of MC Dropout is visible rather than asserted.

What each one measures, since they are easy to conflate:

- **ECE** (expected calibration error, 15 equal-width bins) isolates *calibration*: when the model
  says 0.8, is it right 80% of the time? It says nothing about whether the model can separate the
  classes at all.
- **Brier score** is a proper scoring rule that mixes calibration *and* discrimination. A model can
  improve its Brier while getting worse-calibrated, and vice versa, which is why both are quoted.
- **AUROC** is reported alongside as a control: averaging over passes should not *cost*
  discrimination. If AUROC drops materially, the dropout rate is too high.

The expected result is that MC averaging **improves ECE and Brier** and leaves AUROC roughly
unchanged. That is MC Dropout's real practical benefit — calibrated confidence, not higher accuracy.
If ECE gets *worse*, report it as such; that is a finding about the dropout rate, not a failure to
hide.

In [ ]:
print("PER-FOLD: deterministic vs MC dropout")
print(MC_FOLD_TABLE.round(4).to_string(index=False))
print("\nmean +/- SD across folds")
for a, b, nm in [("ece_det", "ece_mc", "ECE"), ("brier_det", "brier_mc", "Brier"),
                 ("auroc_det", "auroc_mc", "AUROC"), ("acc_det", "acc_mc", "Accuracy")]:
    d, m = MC_FOLD_TABLE[a], MC_FOLD_TABLE[b]
    arrow = "improved" if (nm in ("ECE", "Brier")) == (m.mean() < d.mean()) else "worse"
    print(f"  {nm:9s} deterministic {d.mean():.4f} +/- {d.std(ddof=1):.4f}   "
          f"MC {m.mean():.4f} +/- {m.std(ddof=1):.4f}   ({arrow})")

print("\n" + "=" * 78)
print(f"POOLED OUT-OF-FOLD  ({len(LABELS)} nodules, {len(np.unique(PATIENT_ID))} patients)"
      + ("   [SMOKE - NOT REPORTABLE]" if IS_SMOKE else ""))
print("=" * 78)

UQ_RESULTS = []
for tag, p in [("deterministic", P_DET), (f"mcdropout_T{CFG['MC_PASSES']}", P_MC)]:
    m = core_metrics(LABELS, p, 0.5, CFG)
    m.update(model=CFG["MODEL"], tag=tag, smoke=IS_SMOKE, fingerprint=FINGERPRINT)
    for key, fn in [("auroc", roc_auc_score),
                    ("ece", lambda a, b: expected_calibration_error(a, b, CFG["N_ECE_BINS"])[0]),
                    ("brier", brier_score_loss)]:
        _, lo, hi = bootstrap_ci(LABELS, p, PATIENT_ID, fn, CFG["N_BOOTSTRAP"], CFG["SEED"])
        m[f"{key}_ci_low"], m[f"{key}_ci_high"] = lo, hi
    UQ_RESULTS.append(m)
    print(f"\n  {tag}")
    print(f"    ECE     {m['ece']:.4f}  [{m['ece_ci_low']:.4f}, {m['ece_ci_high']:.4f}]")
    print(f"    Brier   {m['brier']:.4f}  [{m['brier_ci_low']:.4f}, {m['brier_ci_high']:.4f}]")
    print(f"    AUROC   {m['auroc']:.4f}  [{m['auroc_ci_low']:.4f}, {m['auroc_ci_high']:.4f}]")
    print(f"    acc {m['accuracy']:.4f}  sens {m['recall']:.4f}  spec {m['specificity']:.4f}  "
          f"FN {m['fn']}")

d, mc = UQ_RESULTS[0], UQ_RESULTS[1]
print("\n" + "-" * 78)
print(f"  ECE    {d['ece']:.4f} -> {mc['ece']:.4f}   "
      f"({'improved' if mc['ece'] < d['ece'] else 'WORSE - dropout rate may be too high'})")
print(f"  Brier  {d['brier']:.4f} -> {mc['brier']:.4f}   "
      f"({'improved' if mc['brier'] < d['brier'] else 'worse'})")
_a, _b, _z, _p = delong_roc_test(LABELS, P_MC, P_DET)
print(f"  AUROC  {d['auroc']:.4f} -> {mc['auroc']:.4f}   DeLong p = {_p:.4g}"
      f"  ({'no significant change' if _p >= 0.05 else 'significant change'})")
print("-" * 78)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15.5, 4.2))

# (a) reliability diagram, both predictors
for lab, p, col in [("deterministic", P_DET, "#c95c5c"),
                    (f"MC dropout T={CFG['MC_PASSES']}", P_MC, "#3b6ea5")]:
    ece, bins = expected_calibration_error(LABELS, p, CFG["N_ECE_BINS"])
    xs = [b[0] for b in bins if b[3] > 0]; ys = [b[1] for b in bins if b[3] > 0]
    ax[0].plot(xs, ys, "o-", ms=4, color=col, label=f"{lab}  ECE {ece:.3f}")
ax[0].plot([0, 1], [0, 1], "k:", lw=1)
ax[0].set_xlabel("predicted confidence"); ax[0].set_ylabel("observed accuracy")
ax[0].set_title("Reliability (out-of-fold)"); ax[0].legend(fontsize=8, loc="upper left")

# (b) the decomposition, split by whether the prediction was right
correct = (P_MC >= 0.5).astype(int) == LABELS
for msk, lab, col in [(correct, "correct", "#5aa469"), (~correct, "incorrect", "#c95c5c")]:
    ax[1].scatter(OOF["aleatoric"][msk], OOF["epistemic"][msk], s=15, alpha=.6,
                  label=lab, color=col)
ax[1].set_xlabel("aleatoric (image ambiguity)"); ax[1].set_ylabel("epistemic (model ignorance)")
ax[1].set_title("Uncertainty decomposition"); ax[1].legend(fontsize=8)

# (c) the question that decides whether the signal is usable at all
ax[2].hist([OOF["predictive_entropy"][correct], OOF["predictive_entropy"][~correct]],
           bins=20, density=True, color=["#5aa469", "#c95c5c"],
           label=["correct", "incorrect"])
ax[2].set_xlabel("predictive entropy (normalised)"); ax[2].set_ylabel("density")
ax[2].set_title("Uncertainty on correct vs incorrect"); ax[2].legend(fontsize=8)

fig.suptitle(f"{CFG['MODEL']} MC Dropout - out-of-fold over {CFG['N_FOLDS']} folds"
             + ("   [SMOKE]" if IS_SMOKE else ""))
plt.tight_layout(); plt.savefig(f"{CFG['OUT_DIR']}/{CFG['MODEL']}_uq_calibration.png", dpi=110)
plt.show()

ERROR_RANK_AUROC = float(roc_auc_score((~correct).astype(int), OOF["predictive_entropy"]))
print(f"Uncertainty ranks the model's own errors with AUROC = {ERROR_RANK_AUROC:.4f}")
print("  0.5  = uncertainty is noise and must not drive routing")
print("  >0.7 = usable signal for selective prediction / escalation")

## 6.4 &nbsp;Selective prediction - is the uncertainty worth acting on?

Sort nodules by uncertainty, hand the most uncertain fraction to the next stage, and measure error
on what remains. If the curve falls as coverage drops, the estimate has value; if it is flat, acting
on it only adds latency.

The **random-referral control** is plotted alongside, and it is the part that makes this honest:
referring a random subset also produces a downward-looking curve on any imbalanced problem. The gap
between the two curves is the actual finding, summarised as **AURC** (lower is better).

The false-negative curve is plotted separately because a missed cancer is not symmetric with a false
alarm, and the overall error rate hides that.

In [ ]:
def risk_coverage(y, p, uncertainty, threshold=0.5, n_points=50):
    """Ascending uncertainty; at each coverage report error rate and false-negative rate."""
    order = np.argsort(uncertainty)
    y_s, pred_s = y[order], (p[order] >= threshold).astype(int)
    n = len(y)
    covs, errs, fnrs = [], [], []
    for c in np.linspace(max(0.1, 5.0 / n), 1.0, n_points):
        k = max(1, int(round(c * n)))
        yy, pp = y_s[:k], pred_s[:k]
        covs.append(k / n)
        errs.append(float((yy != pp).mean()))
        pos = yy == 1
        fnrs.append(float(((pp == 0) & pos).sum() / max(pos.sum(), 1)))
    covs, errs, fnrs = map(np.asarray, (covs, errs, fnrs))
    integrate = getattr(np, "trapezoid", None) or np.trapz     # np.trapz removed in numpy 2
    return covs, errs, fnrs, float(integrate(errs, covs) / (covs[-1] - covs[0]))


U = OOF["predictive_entropy"]
covs, errs, fnrs, AURC = risk_coverage(LABELS, P_MC, U)

rng = np.random.default_rng(CFG["SEED"])
rand_curves = np.array([risk_coverage(LABELS, P_MC, rng.permutation(U))[1] for _ in range(30)])
rand_mean, aurc_rand = rand_curves.mean(0), None
integrate = getattr(np, "trapezoid", None) or np.trapz
aurc_rand = float(integrate(rand_mean, covs) / (covs[-1] - covs[0]))

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.0))
ax[0].plot(covs, errs, lw=2, color="#3b6ea5", label=f"by uncertainty (AURC {AURC:.4f})")
ax[0].plot(covs, rand_mean, ls=":", lw=1.4, color="grey",
           label=f"random referral (AURC {aurc_rand:.4f})")
ax[0].set_xlabel("coverage (fraction auto-decided)"); ax[0].set_ylabel("error rate")
ax[0].set_title("Risk-coverage"); ax[0].legend(fontsize=8)
ax[1].plot(covs, fnrs, lw=2, color="#c95c5c")
ax[1].set_xlabel("coverage"); ax[1].set_ylabel("false-negative rate")
ax[1].set_title("False negatives vs coverage")
fig.suptitle("Selective prediction" + ("   [SMOKE]" if IS_SMOKE else ""))
plt.tight_layout(); plt.savefig(f"{CFG['OUT_DIR']}/{CFG['MODEL']}_risk_coverage.png", dpi=110)
plt.show()

print(f"AURC by uncertainty {AURC:.4f}  vs random referral {aurc_rand:.4f}  "
      f"(gain {aurc_rand - AURC:+.4f})")
print("\nerror and false negatives at selected coverages")
for c in [1.0, 0.9, 0.8, 0.7]:
    k = int(round(c * len(LABELS))); o = np.argsort(U)[:k]
    err = float((((P_MC[o] >= 0.5).astype(int)) != LABELS[o]).mean())
    fn = int(((P_MC[o] < 0.5) & (LABELS[o] == 1)).sum())
    print(f"  coverage {c:5.0%}  error {err:.4f}   FN {fn:3d}   "
          f"({len(LABELS)-k} nodules referred onward)")

## 6.5 &nbsp;Does the decomposition mean what it claims?

This is the analysis worth writing up, because it tests the theory rather than assuming it.

LIDC gives 3-4 independent malignancy ratings per nodule, so `mal_std` — their standard deviation —
is **measured human ambiguity**. If the model's **aleatoric** term correlates with it while the
**epistemic** term does not, the decomposition is behaving as the theory says: aleatoric tracks
ambiguity that is genuinely in the image, and epistemic tracks something else entirely (the model's
own ignorance).

That is a far stronger claim than "we applied MC dropout", and it is cheap: the ratings are already
cached.

In [ ]:
if len(META) and "mal_std" in META.columns:
    reader_std = META["mal_std"].to_numpy(dtype=float)
    assert len(reader_std) == len(LABELS), "metadata rows must align with the cache"

    if np.std(reader_std) < 1e-9:
        print("No variation in reader disagreement in this cache - analysis not applicable.")
    else:
        print(f"Spearman correlation with radiologist disagreement (mal_std), "
              f"n={len(reader_std)}")
        rows = []
        for key in ["aleatoric", "epistemic", "predictive_entropy", "std_positive"]:
            rho, pv = stats.spearmanr(reader_std, OOF[key])
            rows.append(dict(measure=key, spearman_rho=rho, p_value=pv))
            print(f"  {key:20s} rho = {rho:+.3f}   p = {pv:.4g}")
        READER_CORR = pd.DataFrame(rows)
        READER_CORR.to_csv(f"{CFG['OUT_DIR']}/{CFG['MODEL']}_reader_agreement.csv", index=False)

        r_al = READER_CORR.loc[READER_CORR.measure == "aleatoric", "spearman_rho"].iloc[0]
        r_ep = READER_CORR.loc[READER_CORR.measure == "epistemic", "spearman_rho"].iloc[0]
        print(f"\n  aleatoric rho ({r_al:+.3f}) vs epistemic rho ({r_ep:+.3f}): "
              + ("as predicted - aleatoric tracks human ambiguity more strongly"
                 if r_al > r_ep else
                 "NOT as predicted - report this honestly; it weakens the decomposition claim"))

        fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.8))
        for a, key, col in [(ax[0], "aleatoric", "#b07b2a"), (ax[1], "epistemic", "#2d5f8b")]:
            a.scatter(reader_std, OOF[key], s=16, alpha=.55, color=col)
            a.set_xlabel("radiologist disagreement (SD of malignancy ratings)")
            a.set_ylabel(key)
            rho = READER_CORR.loc[READER_CORR.measure == key, "spearman_rho"].iloc[0]
            a.set_title(f"{key}  (rho {rho:+.3f})")
        fig.suptitle("Model uncertainty vs measured human ambiguity"
                     + ("   [SMOKE]" if IS_SMOKE else ""))
        plt.tight_layout()
        plt.savefig(f"{CFG['OUT_DIR']}/{CFG['MODEL']}_reader_agreement.png", dpi=110); plt.show()
else:
    READER_CORR = pd.DataFrame()
    print("Reader-disagreement analysis needs the LIDC metadata (mal_std), which is absent in "
          "SMOKE/phantom mode.")

## 6.6 &nbsp;Interface for the orchestrator

This is the deliverable to send Dr. Sattar, so the contract is stated explicitly rather than left to
be discovered by reading code.

### One distinction that has to be got right

There are **two different jobs** here, and they need different models:

| Job | Which model | Why |
|---|---|---|
| **Evaluating** uncertainty (everything above) | the fold model that never saw this nodule's patient | trained-on data always looks confident, so anything else inflates the result |
| **Serving** a new, unseen patient | all 5 fold models ensembled | there is no single "final" model after cross-validation, and averaging them is both standard and better-calibrated |

So the numbers reported above come from per-fold out-of-fold inference, while `NoduleUQService`
below ensembles the folds. Worth being aware of the consequence: in ensemble mode the epistemic term
now captures **both** dropout variance and disagreement between fold models. That is a legitimate —
arguably better — epistemic estimate, but it is not the same quantity as the out-of-fold numbers, so
the two should not be quoted interchangeably. `UQ_SERVICE_MODE = "single_fold"` restores strict
comparability if that matters more than serving quality.

### Design decisions worth agreeing on before either side writes more code

- **Versioned.** `schema_version` is in every record, so his side can reject a version it does not
  understand rather than silently misreading a field.
- **Both uncertainty types exposed, not one confidence scalar.** The orchestrator needs the
  epistemic/aleatoric split to decide *how* to escalate. Collapsing them would move that decision
  into this module, where it does not belong.
- **This module recommends; the orchestrator decides.** `routing_hint` is advisory and the
  thresholds used are returned so they can be overridden. Policy is his.
- **Thresholds calibrated on validation, never on the out-of-fold set** — otherwise every number in
  6.3 is invalid.
- **`model_id` and `n_forward_passes` reported**, so a logged decision stays reproducible.

In [ ]:
UQ_SCHEMA_VERSION = "uq-1.1"          # 1.1: fold-ensemble serving, fingerprint in the record

def calibrate_uq_thresholds(val, target_sensitivity, max_referral_rate=0.30):
    """Pick the decision and abstention thresholds on the pooled VALIDATION predictions.

    decision_threshold   : the highest p at which validation sensitivity still meets the target.
                           A screening tool should err towards recall; 0.5 is an arbitrary default,
                           not a clinical choice.
    uncertainty_threshold: the smallest referral rate that removes at least half of the residual
                           false negatives, capped by max_referral_rate - referring everything
                           trivially satisfies any target and helps nobody.
    """
    y, p, u = val["y"], val["mean_prob_positive"], val["predictive_entropy"]

    cand = np.unique(np.round(np.linspace(0.02, 0.98, 97), 4))
    sens = np.array([recall_score(y, (p >= t).astype(int), zero_division=0) for t in cand])
    ok = np.where(sens >= target_sensitivity)[0]
    decision_threshold = float(cand[ok[-1]]) if len(ok) else 0.5

    pred = (p >= decision_threshold).astype(int)
    fn_mask = (pred == 0) & (y == 1)
    base_fn = int(fn_mask.sum())

    best_q, best_rate = float(1.0 - max_referral_rate), float(max_referral_rate)
    for q in np.arange(0.95, 1.0 - max_referral_rate - 1e-9, -0.01):
        thr = float(np.quantile(u, q))
        keep = u <= thr
        if base_fn == 0 or int((fn_mask & keep).sum()) <= base_fn * 0.5:
            best_q, best_rate = float(q), float((~keep).mean())
            break

    return dict(
        decision_threshold    = decision_threshold,
        uncertainty_threshold = float(np.quantile(u, best_q)),
        epistemic_threshold   = float(np.quantile(val["epistemic"], best_q)),
        calibrated_on         = "pooled per-fold validation slices",
        target_sensitivity    = float(target_sensitivity),
        val_sensitivity_at_threshold = float(recall_score(y, pred, zero_division=0)),
        expected_referral_rate = best_rate,
        val_false_negatives_before_referral = base_fn,
    )


class NoduleUQService:
    """Uncertainty-aware inference endpoint - the orchestrator's only entry point.

        svc = NoduleUQService(fold_checkpoints, thresholds)
        records = svc.predict(patches, ids)      # patches: (B, K, H, W) uint8, HU-windowed
        json.dumps(records)                      # every record is JSON-serialisable
    """

    def __init__(self, ckpts_by_fold, thresholds, n_passes=None, cfg=CFG,
                 mode="ensemble", fingerprint=None):
        self.mode = mode
        self.cfg = cfg
        self.n_passes = int(n_passes or cfg["MC_PASSES"])
        self.thresholds = dict(thresholds)
        self.fingerprint = fingerprint
        folds = sorted(ckpts_by_fold) if mode == "ensemble" else [sorted(ckpts_by_fold)[0]]

        self.models, self.folds = [], folds
        for k in folds:
            m, ck = load_fold_model(ckpts_by_fold[k], cfg)
            self.models.append(m)
            self.model_name = ck["name"]
        self.model_id = (f"{self.model_name}|folds={','.join(map(str, folds))}"
                         f"|T={self.n_passes}|mode={mode}")
        self.build_cfg = {k: cfg[k] for k in ["PATCH_MM", "IMG_SIZE", "N_SLICES", "HU_WINDOW",
                                              "MIN_READERS", "DROP_SCORE_3"]}

    def _prepare(self, patches):
        x = torch.as_tensor(np.asarray(patches), dtype=torch.float32)
        if x.ndim == 3:
            x = x.unsqueeze(0)
        if x.max() > 1.5:
            x = x / 255.0
        if x.shape[1] == 1:
            x = x.repeat(1, 3, 1, 1)
        elif x.shape[1] > 3:
            mid = x.shape[1] // 2
            x = x[:, mid - 1: mid + 2]
        return (x - IMAGENET_MEAN.unsqueeze(0)) / IMAGENET_STD.unsqueeze(0)

    def _route(self, entropy, epistemic):
        t = self.thresholds
        if entropy <= t["uncertainty_threshold"]:
            return "accept", False
        # Uncertain. WHICH KIND of uncertainty decides where it goes.
        if epistemic >= t["epistemic_threshold"]:
            return "escalate_model", True    # model ignorance -> another model / TTA / ensemble
        return "human_review", True          # irreducible ambiguity -> escalation wastes compute

    @torch.no_grad()
    def predict(self, patches, ids=None):
        x = self._prepare(patches).to(DEVICE)
        t0 = time.time()
        passes = []
        for m in self.models:                     # every fold model x every MC pass
            enable_mc_dropout(m)
            for _ in range(self.n_passes):
                passes.append(torch.softmax(m(x).float(), dim=1).cpu().numpy())
            m.eval()
        probs = np.stack(passes, axis=0)
        latency_ms = (time.time() - t0) * 1000.0 / max(len(x), 1)

        u = uncertainty_from_passes(probs)
        thr = self.thresholds["decision_threshold"]
        out = []
        for i in range(probs.shape[1]):
            p_pos = float(u["mean_prob_positive"][i])
            label_index = int(p_pos >= thr)
            hint, abstain = self._route(float(u["predictive_entropy"][i]),
                                        float(u["epistemic"][i]))
            out.append(dict(
                schema_version = UQ_SCHEMA_VERSION,
                nodule_id      = (str(ids[i]) if ids is not None else f"idx_{i}"),
                model_id       = self.model_id,
                dataset_fingerprint = self.fingerprint,
                label          = CLASS_NAMES[label_index],
                label_index    = label_index,
                probabilities  = {CLASS_NAMES[0]: round(1.0 - p_pos, 6),
                                  CLASS_NAMES[1]: round(p_pos, 6)},
                confidence     = round(float(u["confidence"][i]), 6),
                uncertainty    = dict(
                    predictive_entropy = round(float(u["predictive_entropy"][i]), 6),
                    aleatoric          = round(float(u["aleatoric"][i]), 6),
                    epistemic          = round(float(u["epistemic"][i]), 6),
                    std_positive       = round(float(u["std_positive"][i]), 6),
                    vote_disagreement  = round(float(u["vote_disagreement"][i]), 6),
                ),
                n_forward_passes = int(probs.shape[0]),
                n_fold_models    = len(self.models),
                abstain          = bool(abstain),
                routing_hint     = hint,
                thresholds       = {k: v for k, v in self.thresholds.items()
                                    if isinstance(v, (int, float))},
                preprocessing    = self.build_cfg,
                latency_ms       = round(latency_ms, 2),
                smoke            = bool(IS_SMOKE),
            ))
        return out

In [ ]:
UQ_THRESHOLDS = calibrate_uq_thresholds(VAL, CFG["UQ_TARGET_SENSITIVITY"])
print("Thresholds (calibrated on pooled validation, never on the out-of-fold set):")
for k, v in UQ_THRESHOLDS.items():
    print(f"  {k:38s} {v}")

# ---- what the policy would do to the out-of-fold set ----
thr, u, ep = UQ_THRESHOLDS, OOF["predictive_entropy"], OOF["epistemic"]
pred = (P_MC >= thr["decision_threshold"]).astype(int)
referred = u > thr["uncertainty_threshold"]
to_model = referred & (ep >= thr["epistemic_threshold"])
kept = ~referred

print(f"\nPolicy applied out-of-fold ({len(LABELS)} nodules):")
print(f"  auto-decided            {kept.sum():4d}  ({kept.mean():.1%})")
print(f"  -> escalate_model       {to_model.sum():4d}")
print(f"  -> human_review         {(referred & ~to_model).sum():4d}")
print(f"  FN among auto-decided   {int(((pred == 0) & (LABELS == 1) & kept).sum()):4d}")
print(f"  FN if nothing referred  {int(((pred == 0) & (LABELS == 1)).sum()):4d}")
if kept.sum() and len(np.unique(LABELS[kept])) > 1:
    print(f"  sensitivity on auto-decided  "
          f"{recall_score(LABELS[kept], pred[kept], zero_division=0):.4f}")
    print(f"  specificity on auto-decided  "
          f"{recall_score(1 - LABELS[kept], 1 - pred[kept], zero_division=0):.4f}")

POLICY = dict(auto_decided=int(kept.sum()), escalate_model=int(to_model.sum()),
              human_review=int((referred & ~to_model).sum()),
              fn_auto_decided=int(((pred == 0) & (LABELS == 1) & kept).sum()),
              fn_no_referral=int(((pred == 0) & (LABELS == 1)).sum()),
              coverage=float(kept.mean()))

In [ ]:
# ---- instantiate and demonstrate on real out-of-fold patches ----
UQ_SERVICE_MODE = "ensemble"          # "ensemble" (all folds) | "single_fold"

SVC = NoduleUQService(CKPTS_BY_FOLD, UQ_THRESHOLDS, CFG["MC_PASSES"], CFG,
                      mode=UQ_SERVICE_MODE, fingerprint=FINGERPRINT)
print(f"service model_id: {SVC.model_id}")
print(f"total forward passes per nodule: {len(SVC.models)} folds x {SVC.n_passes} passes "
      f"= {len(SVC.models) * SVC.n_passes}")

demo_j = np.argsort(-OOF["predictive_entropy"])[:3]        # the 3 most uncertain nodules
demo = SVC.predict(PATCHES[demo_j], ids=[f"{PATIENT_ID[k]}#{k}" for k in demo_j])
print("\nExample record (most uncertain out-of-fold nodule):")
print(json.dumps(demo[0], indent=2))
print("\nRouting hints for the 3 most uncertain nodules:",
      [r["routing_hint"] for r in demo])

---

## 6.7 &nbsp;Results bundle

Per-nodule uncertainty, the calibration numbers, the calibrated thresholds, and the machine-readable
interface contract. Send `uq_interface_contract.json` to Dr. Sattar — the file, not a description of
it — and keep `<model>_mc_oof.npz` for the UQ comparison.

In [ ]:
MODEL = CFG["MODEL"]

# ---- per-nodule uncertainty, for the UQ comparison ----
np.savez_compressed(
    f"{CFG['OUT_DIR']}/{MODEL}_mc_oof.npz",
    p_det=P_DET, p_mc=P_MC, y=LABELS,
    cache_index=np.arange(len(LABELS)),
    patient_id=PATIENT_ID.astype(object),
    fold=OOF["fold"], malignancy=MALIGNANCY,
    **{k: OOF[k] for k in UNC_KEYS},
    model=MODEL, fingerprint=FINGERPRINT, mc_passes=CFG["MC_PASSES"],
)

UQ_SUMMARY = pd.DataFrame(UQ_RESULTS)
keep = ["model", "tag", "n", "ece", "ece_ci_low", "ece_ci_high", "brier", "brier_ci_low",
        "brier_ci_high", "auroc", "auroc_ci_low", "auroc_ci_high", "accuracy", "recall",
        "specificity", "f1", "tp", "fp", "tn", "fn", "smoke"]
UQ_SUMMARY[[c for c in keep if c in UQ_SUMMARY.columns]].to_csv(
    f"{CFG['OUT_DIR']}/{MODEL}_uq_metrics.csv", index=False)
MC_FOLD_TABLE.to_csv(f"{CFG['OUT_DIR']}/{MODEL}_uq_fold_metrics.csv", index=False)

UQ_CONTRACT = {
    "schema_version": UQ_SCHEMA_VERSION,
    "produced_by": f"Task 6 - MC Dropout UQ on the {MODEL} CNN baseline",
    "task": "binary lung nodule malignancy classification (LIDC-IDRI)",
    "dataset_fingerprint": FINGERPRINT,
    "entry_point": "NoduleUQService.predict(patches, ids) -> list[record]",
    "serving_mode": dict(
        mode=UQ_SERVICE_MODE, n_fold_models=len(SVC.models),
        passes_per_fold=CFG["MC_PASSES"],
        total_forward_passes_per_nodule=len(SVC.models) * CFG["MC_PASSES"],
        note=("Evaluation numbers below are per-fold OUT-OF-FOLD. Serving ensembles all folds, so "
              "its epistemic term also includes between-fold disagreement. Do not quote the two "
              "interchangeably."),
    ),
    "input": {
        "patches": {
            "type": "uint8 array", "shape": "(batch, n_slices, H, W)",
            "n_slices": CFG["N_SLICES"], "H_W": CFG["IMG_SIZE"],
            "preprocessing_required": (
                f"crop {CFG['PATCH_MM']}mm x {CFG['PATCH_MM']}mm of anatomy centred on the nodule "
                f"centroid, HU-window to {list(CFG['HU_WINDOW'])}, scale to 0-255, resample to "
                f"{CFG['IMG_SIZE']}x{CFG['IMG_SIZE']}"),
            "note": ("mm-based cropping is mandatory. A fixed-pixel crop makes patches physically "
                     "inconsistent across scanners and silently degrades accuracy."),
        },
        "ids": "optional list of stable nodule identifiers, echoed back as nodule_id",
    },
    "record_fields": {
        "schema_version": "str - reject versions you do not support",
        "nodule_id": "str",
        "model_id": "str - arch|folds|T|mode, for reproducing a logged decision",
        "dataset_fingerprint": "str - which data/fold partition the models were trained on",
        "label": f"str - one of {CLASS_NAMES}",
        "label_index": "int - 0 benign, 1 malignant",
        "probabilities": "dict[str,float] - averaged over all passes, sums to 1",
        "confidence": "float [0,1] - max averaged class probability",
        "uncertainty.predictive_entropy": "float [0,1] - TOTAL uncertainty (normalised by ln2)",
        "uncertainty.aleatoric": "float [0,1] - image ambiguity; escalation will NOT help",
        "uncertainty.epistemic": "float [0,1] - model ignorance; escalation MAY help",
        "uncertainty.std_positive": "float - std of P(malignant) across passes",
        "uncertainty.vote_disagreement": "float [0,1] - 0 all passes agree, 1 even split",
        "n_forward_passes": "int - folds x passes",
        "n_fold_models": "int",
        "abstain": "bool - this module declines to be the final decision",
        "routing_hint": "str - accept | escalate_model | human_review (ADVISORY ONLY)",
        "thresholds": "dict - the calibrated values used, so you can override them",
        "preprocessing": "dict - the exact patch config the models were trained on",
        "latency_ms": "float - per nodule, all passes included",
        "smoke": "bool - TRUE MEANS SYNTHETIC DATA; ignore the numbers entirely",
    },
    "routing_semantics": {
        "accept": "entropy below threshold - use label as the decision",
        "escalate_model": ("uncertain AND epistemic-dominated - try another model, TTA or an "
                           "ensemble; this is where dynamic model selection pays off"),
        "human_review": ("uncertain AND aleatoric-dominated - genuinely ambiguous lesion, more "
                         "compute will not resolve it, route to a radiologist"),
    },
    "guarantees": [
        "thresholds calibrated on validation only, never on the out-of-fold evaluation set",
        "this module recommends; the orchestrator decides",
        "every record is JSON-serialisable with no numpy scalars",
        "identical input + identical seed reproduces identical output",
    ],
    "known_limitations": [
        "binary benign/malignant; the ordinal 1-5 task (proposal item 7) is not implemented",
        f"median malignancy == 3 nodules are EXCLUDED (DROP_SCORE_3={CFG['DROP_SCORE_3']}), so the "
        f"models have never seen an indeterminate lesion and behaviour on one is unvalidated",
        f"nodules require >= {CFG['MIN_READERS']} radiologist marks",
        f"MC Dropout costs {len(SVC.models) * CFG['MC_PASSES']}x a single forward pass in "
        f"{UQ_SERVICE_MODE} mode; budget for it",
        f"evaluated on {len(LABELS)} nodules from {len(np.unique(PATIENT_ID))} patients - "
        f"confidence intervals are wide at this size",
    ],
    "calibrated_thresholds": UQ_THRESHOLDS,
    "policy_on_out_of_fold": POLICY,
    "calibration": {r["tag"]: dict(ece=r["ece"], brier=r["brier"], auroc=r["auroc"],
                                   accuracy=r["accuracy"], recall=r["recall"], fn=r["fn"])
                    for r in UQ_RESULTS},
    "uncertainty_error_ranking_auroc": ERROR_RANK_AUROC,
    "aurc_by_uncertainty": AURC,
    "aurc_random_referral": aurc_rand,
    "reader_agreement_spearman": (READER_CORR.to_dict("records") if len(READER_CORR) else []),
    "data_is_synthetic_smoke": bool(IS_SMOKE),
}

with open(f"{CFG['OUT_DIR']}/uq_interface_contract.json", "w") as fh:
    json.dump(UQ_CONTRACT, fh, indent=2, default=str)

print(f"BUNDLE -> {CFG['OUT_DIR']}")
for p in sorted(glob.glob(f"{CFG['OUT_DIR']}/*")):
    print(f"  {os.path.getsize(p)/1024:9.1f} KB  {os.path.basename(p)}")

d, mc = UQ_RESULTS[0], UQ_RESULTS[1]
print(f"""
{'='*78}
TASK 6  |  MC Dropout on {MODEL}  |  T={CFG['MC_PASSES']}  |  fingerprint {FINGERPRINT}
{'='*78}
  CALIBRATION (the required metrics), pooled out-of-fold
    ECE     deterministic {d['ece']:.4f}   ->   MC {mc['ece']:.4f}
    Brier   deterministic {d['brier']:.4f}   ->   MC {mc['brier']:.4f}
    AUROC   deterministic {d['auroc']:.4f}   ->   MC {mc['auroc']:.4f}   (control)

  UNCERTAINTY
    ranks its own errors with AUROC   {ERROR_RANK_AUROC:.4f}
    AURC by uncertainty               {AURC:.4f}   vs random referral {aurc_rand:.4f}
    auto-decided at policy thresholds {POLICY['coverage']:.1%}

CHECKLIST
  [{'X' if not IS_SMOKE else ' '}] real LIDC data (SMOKE is currently {IS_SMOKE})
  [{'X' if ERROR_RANK_AUROC > 0.7 else ' '}] uncertainty ranks errors above 0.70 (usable for routing)
  [{'X' if AURC < aurc_rand else ' '}] beats random referral
  [ ] uq_interface_contract.json sent to Dr. Sattar
{'='*78}
""")